In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np

file_path = "GSE115943_serum_IDEAS_with_fates_CCAT.h5ad"

adata = sc.read_h5ad(file_path)

print(adata)

AnnData object with n_obs × n_vars = 181124 × 27998
    obs: 'n_genes', 'n_counts', 'timepoint', 'condition', 'KS_score', 'KS_epi_detected', 'KS_mes_detected', 'GS76_score', 'score_hallmark_emt', 'score_emt_core', 'score_met_epithelial', 'score_pluripotency', 'score_fibroblast_id', 'score_bcell_id', 'score_epithelial_id', 'score_pluripotent_fate', 'score_stromal_fate', 'score_trophoblast_fate', 'score_neural_fate', 'score_ks_epithelial', 'score_ks_mesenchymal', 'orig_barcode', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'pct_counts_ribo', 'tp', 'IDEAS_score', 'w_iPSC', 'w_Stromal', 'w_Trophoblast', 'w_Neural', 'P_iPSC', 'P_Stromal', 'P_Trophoblast', 'P_Neural', 'P_Other', 'CCAT_score'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mt', 'ribo', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'
    uns: 'EMT_scoring', 'SCENT_CCAT', 'hvg', 'log1p', 'neighbors', 'pca', 'tp_colors', 'umap'


In [ ]:
from scipy.stats import pearsonr, spearmanr

# Extract the two scores
df_corr = adata.obs[["KS_score", "GS76_score"]].dropna()

# Pearson correlation
pearson_r, pearson_p = pearsonr(
    df_corr["KS_score"],
    df_corr["GS76_score"]
)

# Spearman correlation
spearman_r, spearman_p = spearmanr(
    df_corr["KS_score"],
    df_corr["GS76_score"]
)

print(f"Number of cells: {len(df_corr)}")

print(f"\nPearson correlation:")
print(f"r = {pearson_r:.4f}")
print(f"p = {pearson_p:.3e}")

print(f"\nSpearman correlation:")
print(f"rho = {spearman_r:.4f}")
print(f"p = {spearman_p:.3e}")

Number of cells: 181124

Pearson correlation:
r = -0.6063
p = 0.000e+00

Spearman correlation:
rho = -0.6009
p = 0.000e+00


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 6))

plt.scatter(
    df_corr["KS_score"],
    df_corr["GS76_score"],
    s=3,
    alpha=0.15
)

plt.xlabel("KS score")
plt.ylabel("GS76 score")
plt.title(
    f"KS score vs GS76 score\n"
    f"Pearson r = {pearson_r:.3f}"
)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kruskal
import scikit_posthocs as sp

# -----------------------------
# Prepare data
# -----------------------------

df = adata.obs[
    ["timepoint", "IDEAS_score"]
].copy()

df["day"] = pd.to_numeric(
    df["timepoint"].astype(str),
    errors="coerce"
)

df = df[
    (df["day"] >= 0) &
    (df["day"] <= 7.5)
].dropna(
    subset=["day", "IDEAS_score"]
)

time_order = sorted(df["day"].unique())

# -----------------------------
# Kruskal-Wallis
# -----------------------------

groups = [
    df.loc[df["day"] == day, "IDEAS_score"].values
    for day in time_order
]

H, p = kruskal(*groups)

print("Kruskal-Wallis test")
print(f"H = {H:.3f}")
print(f"p = {p:.3e}")

# -----------------------------
# Dunn's post-hoc test
# -----------------------------

dunn = sp.posthoc_dunn(
    df,
    val_col="IDEAS_score",
    group_col="day",
    p_adjust="fdr_bh"
)

dunn = dunn.loc[time_order, time_order]

print("\nDunn post-hoc test (BH/FDR corrected):")
print(dunn)

# -----------------------------
# Publication-style box plot
# -----------------------------

plt.figure(figsize=(11, 6))

ax = sns.boxplot(
    data=df,
    x="day",
    y="IDEAS_score",
    order=time_order,
    width=0.65,
    showfliers=False,
    linewidth=1.2
)

ax.set_xlabel("Timepoint (days)", fontsize=13)
ax.set_ylabel("IDEAS score", fontsize=13)
ax.set_title(
    "IDEAS score across reprogramming time",
    fontsize=14
)

ax.tick_params(
    axis="both",
    labelsize=11
)

sns.despine()

plt.tight_layout()
plt.show()

Kruskal-Wallis test
H = 30535.204
p = 0.000e+00

Dunn post-hoc test (BH/FDR corrected):
               0.0            0.5            1.0            1.5  \
0.0   1.000000e+00   7.066766e-01   0.000000e+00   4.991255e-80   
0.5   7.066766e-01   1.000000e+00   0.000000e+00   2.794541e-75   
1.0   0.000000e+00   0.000000e+00   1.000000e+00   3.351099e-58   
1.5   4.991255e-80   2.794541e-75   3.351099e-58   1.000000e+00   
2.0   0.000000e+00   0.000000e+00   3.161135e-38  1.534504e-174   
2.5   0.000000e+00   0.000000e+00   2.039088e-02   1.129845e-83   
3.0  3.822140e-186  5.869518e-160   2.147578e-88   9.405738e-02   
3.5   0.000000e+00   0.000000e+00   2.150482e-24   1.085893e-22   
4.0  1.114596e-206  1.723173e-174   8.551781e-96   7.725068e-02   
4.5   7.367621e-44   5.231382e-39  1.729655e-261   2.075254e-22   
5.0   0.000000e+00   0.000000e+00   6.704905e-19   3.395168e-26   
5.5  1.690914e-217  6.499800e-187   2.772702e-67   2.971484e-04   
6.0   7.247676e-93   4.107924e-73   0.000

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kruskal
import scikit_posthocs as sp

# -----------------------------
# Prepare data
# -----------------------------

df = adata.obs[
    ["timepoint", "IDEAS_score"]
].copy()

df["day"] = pd.to_numeric(
    df["timepoint"].astype(str),
    errors="coerce"
)

# Keep days 0–7.5
df = df[
    (df["day"] >= 0) &
    (df["day"] <= 7.5)
].dropna(
    subset=["day", "IDEAS_score"]
)

# -----------------------------
# Combine two consecutive days
# -----------------------------

df["day_group"] = (df["day"] // 2) * 2

df["day_group"] = df["day_group"].astype(int)

# Labels
df["day_group_label"] = (
    df["day_group"].astype(str)
    + "–"
    + (df["day_group"] + 1).astype(str)
)

order = [
    "0–1",
    "2–3",
    "4–5",
    "6–7"
]

df = df[
    df["day_group_label"].isin(order)
]

# -----------------------------
# Kruskal-Wallis
# -----------------------------

groups = [
    df.loc[
        df["day_group_label"] == group,
        "IDEAS_score"
    ].values
    for group in order
]

H, p = kruskal(*groups)

print("IDEAS — Kruskal-Wallis")
print(f"H = {H:.3f}")
print(f"p = {p:.3e}")

# -----------------------------
# Dunn post-hoc
# -----------------------------

dunn = sp.posthoc_dunn(
    df,
    val_col="IDEAS_score",
    group_col="day_group_label",
    p_adjust="fdr_bh"
)

print("\nDunn test — FDR corrected")
print(dunn.loc[order, order])

# -----------------------------
# Plot
# -----------------------------

plt.figure(figsize=(9, 6))

sns.boxplot(
    data=df,
    x="day_group_label",
    y="IDEAS_score",
    order=order,
    showfliers=False,
    width=0.65,
    linewidth=1.2
)

plt.xlabel("Reprogramming time (days)")
plt.ylabel("IDEAS score")
plt.title("IDEAS score across reprogramming time")

sns.despine()
plt.tight_layout()
plt.show()

IDEAS — Kruskal-Wallis
H = 24652.662
p = 0.000e+00

Dunn test — FDR corrected
               0–1  2–3            4–5  6–7
0–1   1.000000e+00  0.0  3.012322e-101  0.0
2–3   0.000000e+00  1.0   0.000000e+00  0.0
4–5  3.012322e-101  0.0   1.000000e+00  0.0
6–7   0.000000e+00  0.0   0.000000e+00  1.0


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kruskal
import scikit_posthocs as sp

# --------------------------------
# Prepare data
# --------------------------------

df = adata.obs[
    ["timepoint", "IDEAS_score"]
].copy()

df["day"] = pd.to_numeric(
    df["timepoint"].astype(str),
    errors="coerce"
)

# Keep 0–7.5
df = df[
    (df["day"] >= 0) &
    (df["day"] <= 7.5)
].dropna(
    subset=["day", "IDEAS_score"]
)

# --------------------------------
# Combine consecutive timepoints
# --------------------------------

group_map = {
    0.0: "0–0.5",
    0.5: "0–0.5",

    1.0: "1–1.5",
    1.5: "1–1.5",

    2.0: "2–2.5",
    2.5: "2–2.5",

    3.0: "3–3.5",
    3.5: "3–3.5",

    4.0: "4–4.5",
    4.5: "4–4.5",

    5.0: "5–5.5",
    5.5: "5–5.5",

    6.0: "6–6.5",
    6.5: "6–6.5",

    7.0: "7–7.5",
    7.5: "7–7.5"
}

df["time_group"] = df["day"].map(group_map)

order = [
    "0–0.5",
    "1–1.5",
    "2–2.5",
    "3–3.5",
    "4–4.5",
    "5–5.5",
    "6–6.5",
    "7–7.5"
]

# Make sure no values are missing
df = df.dropna(subset=["time_group"])

# --------------------------------
# Kruskal-Wallis
# --------------------------------

groups = [
    df.loc[
        df["time_group"] == group,
        "IDEAS_score"
    ].values
    for group in order
]

H, p = kruskal(*groups)

print("Kruskal-Wallis test")
print(f"H = {H:.3f}")
print(f"p = {p:.3e}")

# --------------------------------
# Dunn post-hoc
# --------------------------------

dunn = sp.posthoc_dunn(
    df,
    val_col="IDEAS_score",
    group_col="time_group",
    p_adjust="fdr_bh"
)

print("\nDunn post-hoc test (FDR corrected)")
print(dunn.loc[order, order])

# --------------------------------
# Plot
# --------------------------------

plt.figure(figsize=(12, 6))

sns.boxplot(
    data=df,
    x="time_group",
    y="IDEAS_score",
    order=order,
    showfliers=False,
    width=0.65,
    linewidth=1.2
)

plt.xlabel("Reprogramming time")
plt.ylabel("IDEAS score")
plt.title("IDEAS score across reprogramming time")

sns.despine()
plt.tight_layout()
plt.show()

Kruskal-Wallis test
H = 29117.146
p = 0.000e+00

Dunn post-hoc test (FDR corrected)
               0–0.5          1–1.5          2–2.5         3–3.5  \
0–0.5   1.000000e+00   0.000000e+00   0.000000e+00  0.000000e+00   
1–1.5   0.000000e+00   1.000000e+00   4.414648e-96  1.592246e-19   
2–2.5   0.000000e+00   4.414648e-96   1.000000e+00  0.000000e+00   
3–3.5   0.000000e+00   1.592246e-19   0.000000e+00  1.000000e+00   
4–4.5  1.020903e-217  3.496392e-134   0.000000e+00  2.955618e-98   
5–5.5   0.000000e+00   5.619888e-12  5.953169e-302  6.099499e-03   
6–6.5  9.816688e-234   0.000000e+00   0.000000e+00  0.000000e+00   
7–7.5  1.390188e-144   0.000000e+00   0.000000e+00  0.000000e+00   

               4–4.5          5–5.5          6–6.5          7–7.5  
0–0.5  1.020903e-217   0.000000e+00  9.816688e-234  1.390188e-144  
1–1.5  3.496392e-134   5.619888e-12   0.000000e+00   0.000000e+00  
2–2.5   0.000000e+00  5.953169e-302   0.000000e+00   0.000000e+00  
3–3.5   2.955618e-98   6.099499

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kruskal
import scikit_posthocs as sp

# -----------------------------
# Prepare data
# -----------------------------

df = adata.obs[
    ["timepoint", "IDEAS_score"]
].copy()

df["day"] = pd.to_numeric(
    df["timepoint"].astype(str),
    errors="coerce"
)

df = df[
    (df["day"] >= 0) &
    (df["day"] <= 7.5)
].dropna(
    subset=["day", "IDEAS_score"]
)

# -----------------------------
# Combine two adjacent timepoints
# -----------------------------

group_map = {
    0.0: "0–0.5",  0.5: "0–0.5",
    1.0: "1–1.5",  1.5: "1–1.5",
    2.0: "2–2.5",  2.5: "2–2.5",
    3.0: "3–3.5",  3.5: "3–3.5",
    4.0: "4–4.5",  4.5: "4–4.5",
    5.0: "5–5.5",  5.5: "5–5.5",
    6.0: "6–6.5",  6.5: "6–6.5",
    7.0: "7–7.5",  7.5: "7–7.5"
}

df["time_group"] = df["day"].map(group_map)

order = [
    "0–0.5",
    "1–1.5",
    "2–2.5",
    "3–3.5",
    "4–4.5",
    "5–5.5",
    "6–6.5",
    "7–7.5"
]

df = df.dropna(subset=["time_group"])

# -----------------------------
# Statistics
# -----------------------------

groups = [
    df.loc[
        df["time_group"] == group,
        "IDEAS_score"
    ].values
    for group in order
]

H, p = kruskal(*groups)

print("IDEAS — Kruskal-Wallis")
print(f"H = {H:.3f}")
print(f"p = {p:.3e}")

dunn = sp.posthoc_dunn(
    df,
    val_col="IDEAS_score",
    group_col="time_group",
    p_adjust="fdr_bh"
)

print("\nDunn post-hoc — FDR corrected")
print(dunn.loc[order, order])

# -----------------------------
# Violin plot
# -----------------------------

plt.figure(figsize=(12, 6))

sns.violinplot(
    data=df,
    x="time_group",
    y="IDEAS_score",
    order=order,
    inner="box",
    cut=0,
    linewidth=1.1
)

plt.xlabel("Reprogramming time (days)", fontsize=13)
plt.ylabel("IDEAS score", fontsize=13)
plt.title("IDEAS score across reprogramming time", fontsize=14)

sns.despine()
plt.tight_layout()
plt.show()

IDEAS — Kruskal-Wallis
H = 29117.146
p = 0.000e+00

Dunn post-hoc — FDR corrected
               0–0.5          1–1.5          2–2.5         3–3.5  \
0–0.5   1.000000e+00   0.000000e+00   0.000000e+00  0.000000e+00   
1–1.5   0.000000e+00   1.000000e+00   4.414648e-96  1.592246e-19   
2–2.5   0.000000e+00   4.414648e-96   1.000000e+00  0.000000e+00   
3–3.5   0.000000e+00   1.592246e-19   0.000000e+00  1.000000e+00   
4–4.5  1.020903e-217  3.496392e-134   0.000000e+00  2.955618e-98   
5–5.5   0.000000e+00   5.619888e-12  5.953169e-302  6.099499e-03   
6–6.5  9.816688e-234   0.000000e+00   0.000000e+00  0.000000e+00   
7–7.5  1.390188e-144   0.000000e+00   0.000000e+00  0.000000e+00   

               4–4.5          5–5.5          6–6.5          7–7.5  
0–0.5  1.020903e-217   0.000000e+00  9.816688e-234  1.390188e-144  
1–1.5  3.496392e-134   5.619888e-12   0.000000e+00   0.000000e+00  
2–2.5   0.000000e+00  5.953169e-302   0.000000e+00   0.000000e+00  
3–3.5   2.955618e-98   6.099499e-

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kruskal
import scikit_posthocs as sp

# -----------------------------
# Prepare data
# -----------------------------

df = adata.obs[
    ["timepoint", "CCAT_score"]
].copy()

df["day"] = pd.to_numeric(
    df["timepoint"].astype(str),
    errors="coerce"
)

df = df[
    (df["day"] >= 0) &
    (df["day"] <= 7.5)
].dropna(
    subset=["day", "CCAT_score"]
)

# -----------------------------
# Combine two adjacent timepoints
# -----------------------------

df["time_group"] = df["day"].map(group_map)

df = df.dropna(subset=["time_group"])

# -----------------------------
# Statistics
# -----------------------------

groups = [
    df.loc[
        df["time_group"] == group,
        "CCAT_score"
    ].values
    for group in order
]

H, p = kruskal(*groups)

print("CCAT — Kruskal-Wallis")
print(f"H = {H:.3f}")
print(f"p = {p:.3e}")

dunn = sp.posthoc_dunn(
    df,
    val_col="CCAT_score",
    group_col="time_group",
    p_adjust="fdr_bh"
)

print("\nDunn post-hoc — FDR corrected")
print(dunn.loc[order, order])

# -----------------------------
# Violin plot
# -----------------------------

plt.figure(figsize=(12, 6))

sns.violinplot(
    data=df,
    x="time_group",
    y="CCAT_score",
    order=order,
    inner="box",
    cut=0,
    linewidth=1.1
)

plt.xlabel("Reprogramming time (days)", fontsize=13)
plt.ylabel("CCAT score", fontsize=13)
plt.title("CCAT score across reprogramming time", fontsize=14)

sns.despine()
plt.tight_layout()
plt.show()

CCAT — Kruskal-Wallis
H = 14176.380
p = 0.000e+00

Dunn post-hoc — FDR corrected
               0–0.5          1–1.5         2–2.5         3–3.5  \
0–0.5   1.000000e+00   0.000000e+00  7.622291e-14  1.153504e-67   
1–1.5   0.000000e+00   1.000000e+00  0.000000e+00  0.000000e+00   
2–2.5   7.622291e-14   0.000000e+00  1.000000e+00  1.082169e-34   
3–3.5   1.153504e-67   0.000000e+00  1.082169e-34  1.000000e+00   
4–4.5  4.591473e-197  1.054499e-215  0.000000e+00  0.000000e+00   
5–5.5  3.172850e-239  3.323482e-163  0.000000e+00  0.000000e+00   
6–6.5   0.000000e+00   2.727807e-85  0.000000e+00  0.000000e+00   
7–7.5   0.000000e+00   2.996126e-07  0.000000e+00  0.000000e+00   

               4–4.5          5–5.5         6–6.5          7–7.5  
0–0.5  4.591473e-197  3.172850e-239  0.000000e+00   0.000000e+00  
1–1.5  1.054499e-215  3.323482e-163  2.727807e-85   2.996126e-07  
2–2.5   0.000000e+00   0.000000e+00  0.000000e+00   0.000000e+00  
3–3.5   0.000000e+00   0.000000e+00  0.000000e+

In [ ]:
df_id = adata.obs[
    ["timepoint", "IDEAS_ID_score"]
].copy()

df_id["day"] = pd.to_numeric(
    df_id["timepoint"].astype(str),
    errors="coerce"
)

# Keep day 0–7.5
df_id = df_id[
    (df_id["day"] >= 0) &
    (df_id["day"] <= 7.5)
].dropna(
    subset=["day", "IDEAS_ID_score"]
)

# Combine the same pairs of timepoints
df_id["time_group"] = df_id["day"].map(group_map)

# Remove anything that didn't map
df_id = df_id.dropna(subset=["time_group"])

print(df_id.head())
print(df_id["IDEAS_ID_score"].describe())

                             timepoint  IDEAS_ID_score  day time_group
AAACCTGAGTATCTCG-1_D0.0_Dox        0.0        0.592773  0.0      0–0.5
AAACCTGCAAGCGAGT-1_D0.0_Dox        0.0        0.586047  0.0      0–0.5
AAACCTGGTCTCTTAT-1_D0.0_Dox        0.0        0.405626  0.0      0–0.5
AAACCTGTCAACACTG-1_D0.0_Dox        0.0        0.566065  0.0      0–0.5
AAACCTGTCGTTGCCT-1_D0.0_Dox        0.0        0.584498  0.0      0–0.5
count    98197.000000
mean         0.603569
std          0.126119
min          0.013191
25%          0.523234
50%          0.613449
75%          0.694199
max          1.000000
Name: IDEAS_ID_score, dtype: float64


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kruskal
import scikit_posthocs as sp

# ============================================================
# 1. PREPARE DATA
# ============================================================

df_id = adata.obs[
    ["timepoint", "IDEAS_ID_score"]
].copy()

df_id["day"] = pd.to_numeric(
    df_id["timepoint"].astype(str),
    errors="coerce"
)

# Keep only 0–7.5 days
df_id = df_id[
    (df_id["day"] >= 0) &
    (df_id["day"] <= 7.5)
].dropna(
    subset=["day", "IDEAS_ID_score"]
)

# ============================================================
# 2. COMBINE ADJACENT TIMEPOINTS
# ============================================================

group_map = {
    0.0: "0–0.5",   0.5: "0–0.5",
    1.0: "1–1.5",   1.5: "1–1.5",
    2.0: "2–2.5",   2.5: "2–2.5",
    3.0: "3–3.5",   3.5: "3–3.5",
    4.0: "4–4.5",   4.5: "4–4.5",
    5.0: "5–5.5",   5.5: "5–5.5",
    6.0: "6–6.5",   6.5: "6–6.5",
    7.0: "7–7.5",   7.5: "7–7.5"
}

order = [
    "0–0.5",
    "1–1.5",
    "2–2.5",
    "3–3.5",
    "4–4.5",
    "5–5.5",
    "6–6.5",
    "7–7.5"
]

df_id["time_group"] = df_id["day"].map(group_map)

df_id = df_id.dropna(
    subset=["time_group"]
)

# ============================================================
# 3. KRUSKAL-WALLIS TEST
# ============================================================

groups = [
    df_id.loc[
        df_id["time_group"] == g,
        "IDEAS_ID_score"
    ].values
    for g in order
]

H, p_kw = kruskal(*groups)

print("Kruskal-Wallis test")
print(f"H = {H:.3f}")
print(f"p = {p_kw:.3e}")

# ============================================================
# 4. DUNN POST-HOC TEST
# ============================================================

dunn_id = sp.posthoc_dunn(
    df_id,
    val_col="IDEAS_ID_score",
    group_col="time_group",
    p_adjust="fdr_bh"
)

dunn_id = dunn_id.loc[order, order]

print("\nDunn test — Benjamini-Hochberg corrected")
print(dunn_id)

# ============================================================
# 5. CELL NUMBERS
# ============================================================

cell_counts = (
    df_id
    .groupby("time_group", observed=True)
    .size()
    .reindex(order)
)

print("\nCell numbers:")
print(cell_counts)

# ============================================================
# 6. SIGNIFICANCE FUNCTION
# ============================================================

def significance_symbol(p):

    if p < 0.0001:
        return "****"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

# ============================================================
# 7. FIGURE
# ============================================================

fig, ax = plt.subplots(
    figsize=(12, 8)
)

# Violin
sns.violinplot(
    data=df_id,
    x="time_group",
    y="IDEAS_ID_score",
    order=order,
    inner=None,
    cut=0,
    linewidth=1.2,
    ax=ax
)

# Box plot inside violin
sns.boxplot(
    data=df_id,
    x="time_group",
    y="IDEAS_ID_score",
    order=order,
    width=0.15,
    showfliers=False,
    boxprops=dict(
        facecolor="white",
        edgecolor="black",
        linewidth=1.1,
        zorder=3
    ),
    whiskerprops=dict(
        color="black",
        linewidth=1.1
    ),
    capprops=dict(
        color="black",
        linewidth=1.1
    ),
    medianprops=dict(
        color="black",
        linewidth=1.5
    ),
    ax=ax
)

# ============================================================
# 8. CELL NUMBERS BELOW X-AXIS
# ============================================================

for i, group in enumerate(order):

    n = cell_counts.loc[group]

    ax.text(
        i,
        -0.075,
        f"n = {n:,}",
        transform=ax.get_xaxis_transform(),
        ha="center",
        va="top",
        fontsize=9
    )

# ============================================================
# 9. ADJACENT DUNN COMPARISONS
# ============================================================

# Start above the violins
base_y = 0.86
step = 0.035

for i in range(len(order) - 1):

    g1 = order[i]
    g2 = order[i + 1]

    p_adj = dunn_id.loc[g1, g2]

    y = base_y + i * step

    # Bracket
    ax.plot(
        [i, i, i + 1, i + 1],
        [y, y + 0.012, y + 0.012, y],
        transform=ax.get_xaxis_transform(),
        color="black",
        linewidth=1
    )

    # Significance
    ax.text(
        i + 0.5,
        y + 0.015,
        significance_symbol(p_adj),
        transform=ax.get_xaxis_transform(),
        ha="center",
        va="bottom",
        fontsize=10
    )

# ============================================================
# 10. OVERALL KRUSKAL-WALLIS RESULT
# ============================================================

if p_kw < 0.001:
    p_text = "p < 0.001"
else:
    p_text = f"p = {p_kw:.3e}"

ax.text(
    0.02,
    0.97,
    f"Kruskal–Wallis\nH = {H:.1f}, {p_text}",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=10,
    bbox=dict(
        facecolor="white",
        edgecolor="black",
        linewidth=0.8,
        boxstyle="round,pad=0.4"
    )
)

# ============================================================
# 11. AXIS LABELS
# ============================================================

ax.set_xlabel(
    "Reprogramming time (days)",
    fontsize=13,
    labelpad=38
)

ax.set_ylabel(
    "IDEAS ID-score",
    fontsize=13
)

# No large title — cleaner for publication
ax.set_title("")

# Y-axis limits
ax.set_ylim(
    0,
    1.08
)

ax.set_yticks(
    np.arange(0, 1.01, 0.2)
)

ax.tick_params(
    axis="both",
    labelsize=11
)

# Remove top/right borders
sns.despine(
    ax=ax
)

# Make space for n values
plt.subplots_adjust(
    bottom=0.23,
    top=0.95,
    left=0.10,
    right=0.98
)

plt.show()

Kruskal-Wallis test
H = 29117.146
p = 0.000e+00

Dunn test — Benjamini-Hochberg corrected
               0–0.5          1–1.5          2–2.5         3–3.5  \
0–0.5   1.000000e+00   0.000000e+00   0.000000e+00  0.000000e+00   
1–1.5   0.000000e+00   1.000000e+00   4.414648e-96  1.592246e-19   
2–2.5   0.000000e+00   4.414648e-96   1.000000e+00  0.000000e+00   
3–3.5   0.000000e+00   1.592246e-19   0.000000e+00  1.000000e+00   
4–4.5  1.020903e-217  3.496392e-134   0.000000e+00  2.955618e-98   
5–5.5   0.000000e+00   5.619888e-12  5.953169e-302  6.099499e-03   
6–6.5  9.816688e-234   0.000000e+00   0.000000e+00  0.000000e+00   
7–7.5  1.390188e-144   0.000000e+00   0.000000e+00  0.000000e+00   

               4–4.5          5–5.5          6–6.5          7–7.5  
0–0.5  1.020903e-217   0.000000e+00  9.816688e-234  1.390188e-144  
1–1.5  3.496392e-134   5.619888e-12   0.000000e+00   0.000000e+00  
2–2.5   0.000000e+00  5.953169e-302   0.000000e+00   0.000000e+00  
3–3.5   2.955618e-98   6.

In [ ]:
# ============================================================
# PUBLICATION-READY IDEAS FIGURE
# ============================================================

fig, ax = plt.subplots(
    figsize=(12, 8)
)

# ------------------------------------------------------------
# Violin
# ------------------------------------------------------------

sns.violinplot(
    data=df_id,
    x="time_group",
    y="IDEAS_ID_score",
    order=order,
    inner=None,
    cut=0,
    linewidth=1.2,
    ax=ax
)

# ------------------------------------------------------------
# Box plot
# ------------------------------------------------------------

sns.boxplot(
    data=df_id,
    x="time_group",
    y="IDEAS_ID_score",
    order=order,
    width=0.15,
    showfliers=False,
    boxprops=dict(
        facecolor="white",
        edgecolor="black",
        linewidth=1.1,
        zorder=3
    ),
    whiskerprops=dict(
        color="black",
        linewidth=1.1
    ),
    capprops=dict(
        color="black",
        linewidth=1.1
    ),
    medianprops=dict(
        color="black",
        linewidth=1.5
    ),
    ax=ax
)

# ============================================================
# CELL NUMBERS
# ============================================================

for i, group in enumerate(order):

    ax.text(
        i,
        -0.075,
        f"n = {cell_counts.loc[group]:,}",
        transform=ax.get_xaxis_transform(),
        ha="center",
        va="top",
        fontsize=9
    )

# ============================================================
# ANOVA RESULT
# ============================================================

p_text = (
    "p < 0.001"
    if p_anova < 0.001
    else f"p = {p_anova:.3e}"
)

ax.text(
    0.02,
    0.97,
    f"One-way ANOVA\nF = {F:.1f}, {p_text}",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=10,
    bbox=dict(
        facecolor="white",
        edgecolor="black",
        linewidth=0.8,
        boxstyle="round,pad=0.4"
    )
)

# ============================================================
# SIGNIFICANCE BRACKETS
# ============================================================

# IMPORTANT:
# Brackets are positioned in AXES coordinates rather than
# IDEAS-score coordinates.
#
# Therefore the y-axis remains exactly 0–1.

bracket_y_start = 1.02
bracket_step = 0.055

for i, p_adj in enumerate(adjacent_p):

    y = bracket_y_start + i * bracket_step

    # bracket
    ax.plot(
        [i, i, i + 1, i + 1],
        [y, y + 0.018, y + 0.018, y],
        transform=ax.get_xaxis_transform(),
        color="black",
        linewidth=1.0,
        clip_on=False
    )

    # significance
    ax.text(
        i + 0.5,
        y + 0.022,
        sig_symbol(p_adj),
        transform=ax.get_xaxis_transform(),
        ha="center",
        va="bottom",
        fontsize=10,
        clip_on=False
    )

# ============================================================
# AXES
# ============================================================

ax.set_xlabel(
    "Reprogramming time (days)",
    fontsize=13,
    labelpad=38
)

ax.set_ylabel(
    "IDEAS ID-score",
    fontsize=13
)

# Keep the IDEAS scale biologically meaningful
ax.set_ylim(
    0,
    1.0
)

ax.set_yticks(
    np.arange(0, 1.01, 0.2)
)

ax.tick_params(
    axis="both",
    labelsize=11
)

ax.set_title("")

sns.despine(ax=ax)

# ============================================================
# CREATE SPACE FOR BRACKETS AND CELL NUMBERS
# ============================================================

plt.subplots_adjust(
    left=0.10,
    right=0.98,
    bottom=0.23,
    top=0.70
)

plt.show()

CCAT — Kruskal-Wallis
H = 14176.380
p = 0.000e+00

Dunn test — Benjamini-Hochberg corrected
               0–0.5          1–1.5         2–2.5         3–3.5  \
0–0.5   1.000000e+00   0.000000e+00  7.622291e-14  1.153504e-67   
1–1.5   0.000000e+00   1.000000e+00  0.000000e+00  0.000000e+00   
2–2.5   7.622291e-14   0.000000e+00  1.000000e+00  1.082169e-34   
3–3.5   1.153504e-67   0.000000e+00  1.082169e-34  1.000000e+00   
4–4.5  4.591473e-197  1.054499e-215  0.000000e+00  0.000000e+00   
5–5.5  3.172850e-239  3.323482e-163  0.000000e+00  0.000000e+00   
6–6.5   0.000000e+00   2.727807e-85  0.000000e+00  0.000000e+00   
7–7.5   0.000000e+00   2.996126e-07  0.000000e+00  0.000000e+00   

               4–4.5          5–5.5         6–6.5          7–7.5  
0–0.5  4.591473e-197  3.172850e-239  0.000000e+00   0.000000e+00  
1–1.5  1.054499e-215  3.323482e-163  2.727807e-85   2.996126e-07  
2–2.5   0.000000e+00   0.000000e+00  0.000000e+00   0.000000e+00  
3–3.5   0.000000e+00   0.000000e+00 

In [ ]:
# ============================================================
# PUBLICATION-READY IDEAS FIGURE
# ============================================================

fig, ax = plt.subplots(
    figsize=(12, 8)
)

# ------------------------------------------------------------
# Violin
# ------------------------------------------------------------

sns.violinplot(
    data=df_id,
    x="time_group",
    y="IDEAS_ID_score",
    order=order,
    inner=None,
    cut=0,
    linewidth=1.2,
    ax=ax
)

# ------------------------------------------------------------
# Box plot
# ------------------------------------------------------------

sns.boxplot(
    data=df_id,
    x="time_group",
    y="IDEAS_ID_score",
    order=order,
    width=0.15,
    showfliers=False,
    boxprops=dict(
        facecolor="white",
        edgecolor="black",
        linewidth=1.1,
        zorder=3
    ),
    whiskerprops=dict(
        color="black",
        linewidth=1.1
    ),
    capprops=dict(
        color="black",
        linewidth=1.1
    ),
    medianprops=dict(
        color="black",
        linewidth=1.5
    ),
    ax=ax
)

# ============================================================
# CELL NUMBERS
# ============================================================

for i, group in enumerate(order):

    ax.text(
        i,
        -0.075,
        f"n = {cell_counts.loc[group]:,}",
        transform=ax.get_xaxis_transform(),
        ha="center",
        va="top",
        fontsize=9
    )

# ============================================================
# ANOVA RESULT
# ============================================================

p_text = (
    "p < 0.001"
    if p_anova < 0.001
    else f"p = {p_anova:.3e}"
)

ax.text(
    0.02,
    0.97,
    f"One-way ANOVA\nF = {F:.1f}, {p_text}",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=10,
    bbox=dict(
        facecolor="white",
        edgecolor="black",
        linewidth=0.8,
        boxstyle="round,pad=0.4"
    )
)

# ============================================================
# SIGNIFICANCE BRACKETS
# ============================================================

# IMPORTANT:
# Brackets are positioned in AXES coordinates rather than
# IDEAS-score coordinates.
#
# Therefore the y-axis remains exactly 0–1.

bracket_y_start = 1.02
bracket_step = 0.055

for i, p_adj in enumerate(adjacent_p):

    y = bracket_y_start + i * bracket_step

    # bracket
    ax.plot(
        [i, i, i + 1, i + 1],
        [y, y + 0.018, y + 0.018, y],
        transform=ax.get_xaxis_transform(),
        color="black",
        linewidth=1.0,
        clip_on=False
    )

    # significance
    ax.text(
        i + 0.5,
        y + 0.022,
        sig_symbol(p_adj),
        transform=ax.get_xaxis_transform(),
        ha="center",
        va="bottom",
        fontsize=10,
        clip_on=False
    )

# ============================================================
# AXES
# ============================================================

ax.set_xlabel(
    "Reprogramming time (days)",
    fontsize=13,
    labelpad=38
)

ax.set_ylabel(
    "IDEAS ID-score",
    fontsize=13
)

# Keep the IDEAS scale biologically meaningful
ax.set_ylim(
    0,
    1.0
)

ax.set_yticks(
    np.arange(0, 1.01, 0.2)
)

ax.tick_params(
    axis="both",
    labelsize=11
)

ax.set_title("")

sns.despine(ax=ax)

# ============================================================
# CREATE SPACE FOR BRACKETS AND CELL NUMBERS
# ============================================================

plt.subplots_adjust(
    left=0.10,
    right=0.98,
    bottom=0.23,
    top=0.70
)

plt.show()

In [ ]:
# ============================================================
# PREPARE CCAT DATA
# ============================================================

df_ccat = adata.obs[
    ["timepoint", "CCAT_score"]
].copy()

df_ccat["day"] = pd.to_numeric(
    df_ccat["timepoint"].astype(str),
    errors="coerce"
)

df_ccat = df_ccat[
    (df_ccat["day"] >= 0) &
    (df_ccat["day"] <= 7.5)
].dropna(
    subset=["day", "CCAT_score"]
)

df_ccat["time_group"] = df_ccat["day"].map(group_map)
df_ccat = df_ccat.dropna(subset=["time_group"])


# ============================================================
# ANOVA
# ============================================================

groups = [
    df_ccat.loc[
        df_ccat["time_group"] == g,
        "CCAT_score"
    ].values
    for g in order
]

F, p_anova = f_oneway(*groups)

print("CCAT — One-way ANOVA")
print(f"F = {F:.3f}")
print(f"p = {p_anova:.3e}")


# ============================================================
# TUKEY HSD
# ============================================================

tukey = pairwise_tukeyhsd(
    endog=df_ccat["CCAT_score"],
    groups=df_ccat["time_group"],
    alpha=0.05
)

tukey_df = pd.DataFrame(
    tukey._results_table.data[1:],
    columns=tukey._results_table.data[0]
)

tukey_df["p-adj"] = tukey_df["p-adj"].astype(float)

print("\nTukey HSD")
print(tukey)


# ============================================================
# CELL COUNTS
# ============================================================

cell_counts = (
    df_ccat
    .groupby("time_group", observed=True)
    .size()
    .reindex(order)
)

print("\nCell counts:")
print(cell_counts)


# ============================================================
# ADJACENT TUKEY P-VALUES
# ============================================================

adjacent_p = []

for i in range(len(order) - 1):

    g1 = order[i]
    g2 = order[i + 1]

    row = tukey_df[
        ((tukey_df["group1"] == g1) &
         (tukey_df["group2"] == g2)) |
        ((tukey_df["group1"] == g2) &
         (tukey_df["group2"] == g1))
    ]

    adjacent_p.append(
        float(row.iloc[0]["p-adj"])
        if len(row) else np.nan
    )


# ============================================================
# PLOT
# ============================================================

fig, ax = plt.subplots(figsize=(12, 8))

sns.violinplot(
    data=df_ccat,
    x="time_group",
    y="CCAT_score",
    order=order,
    inner=None,
    cut=0,
    linewidth=1.2,
    ax=ax
)

sns.boxplot(
    data=df_ccat,
    x="time_group",
    y="CCAT_score",
    order=order,
    width=0.15,
    showfliers=False,
    boxprops=dict(
        facecolor="white",
        edgecolor="black",
        linewidth=1.1,
        zorder=3
    ),
    whiskerprops=dict(color="black"),
    capprops=dict(color="black"),
    medianprops=dict(
        color="black",
        linewidth=1.5
    ),
    ax=ax
)


# ============================================================
# CELL NUMBERS
# ============================================================

for i, group in enumerate(order):

    ax.text(
        i,
        -0.075,
        f"n = {cell_counts.loc[group]:,}",
        transform=ax.get_xaxis_transform(),
        ha="center",
        va="top",
        fontsize=9
    )


# ============================================================
# ANOVA RESULT
# ============================================================

p_text = (
    "p < 0.001"
    if p_anova < 0.001
    else f"p = {p_anova:.3e}"
)

ax.text(
    0.02,
    0.97,
    f"One-way ANOVA\nF = {F:.1f}, {p_text}",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=10,
    bbox=dict(
        facecolor="white",
        edgecolor="black",
        boxstyle="round,pad=0.4"
    )
)


# ============================================================
# ADJACENT TUKEY BRACKETS
# ============================================================

ymin = df_ccat["CCAT_score"].min()
ymax = df_ccat["CCAT_score"].max()
yrange = ymax - ymin

base_y = ymax + 0.06 * yrange
step = 0.055 * yrange

for i, p_adj in enumerate(adjacent_p):

    y = base_y + i * step

    ax.plot(
        [i, i, i+1, i+1],
        [
            y,
            y + 0.012 * yrange,
            y + 0.012 * yrange,
            y
        ],
        color="black",
        linewidth=1
    )

    ax.text(
        i + 0.5,
        y + 0.015 * yrange,
        sig_symbol(p_adj),
        ha="center",
        va="bottom",
        fontsize=10
    )


ax.set_xlabel(
    "Reprogramming time (days)",
    fontsize=13,
    labelpad=38
)

ax.set_ylabel(
    "CCAT score",
    fontsize=13
)

ax.tick_params(labelsize=11)

sns.despine(ax=ax)

plt.subplots_adjust(
    bottom=0.23,
    top=0.95,
    left=0.10,
    right=0.98
)

plt.show()

CCAT — One-way ANOVA
F = 2520.955
p = 0.000e+00

Tukey HSD
Multiple Comparison of Means - Tukey HSD, FWER=0.05
group1 group2 meandiff p-adj  lower   upper  reject
---------------------------------------------------
 0–0.5  1–1.5   0.0382   0.0  0.0363    0.04   True
 0–0.5  2–2.5   -0.005   0.0 -0.0064 -0.0035   True
 0–0.5  3–3.5   -0.009   0.0 -0.0105 -0.0075   True
 0–0.5  4–4.5   0.0149   0.0  0.0134  0.0163   True
 0–0.5  5–5.5    0.017   0.0  0.0155  0.0185   True
 0–0.5  6–6.5   0.0222   0.0  0.0206  0.0237   True
 0–0.5  7–7.5   0.0313   0.0  0.0297  0.0328   True
 1–1.5  2–2.5  -0.0432   0.0 -0.0448 -0.0415   True
 1–1.5  3–3.5  -0.0472   0.0 -0.0488 -0.0455   True
 1–1.5  4–4.5  -0.0233   0.0  -0.025 -0.0217   True
 1–1.5  5–5.5  -0.0212   0.0 -0.0229 -0.0195   True
 1–1.5  6–6.5   -0.016   0.0 -0.0178 -0.0143   True
 1–1.5  7–7.5  -0.0069   0.0 -0.0086 -0.0052   True
 2–2.5  3–3.5   -0.004   0.0 -0.0052 -0.0028   True
 2–2.5  4–4.5   0.0198   0.0  0.0186   0.021   True
 2–2.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr

# ============================================================
# Prepare paired data
# ============================================================

corr_df = adata.obs[
    ["IDEAS_ID_score", "CCAT_score"]
].copy()

corr_df = corr_df.dropna()

print("Number of paired cells:", len(corr_df))


# ============================================================
# Pearson correlation
# ============================================================

pearson_r, pearson_p = pearsonr(
    corr_df["IDEAS_ID_score"],
    corr_df["CCAT_score"]
)


# ============================================================
# Spearman correlation
# ============================================================

spearman_rho, spearman_p = spearmanr(
    corr_df["IDEAS_ID_score"],
    corr_df["CCAT_score"]
)


# ============================================================
# Print results
# ============================================================

print("\nCorrelation between IDEAS ID-score and CCAT")
print("---------------------------------------------")

print(
    f"Pearson r = {pearson_r:.4f}, "
    f"p = {pearson_p:.3e}"
)

print(
    f"Spearman rho = {spearman_rho:.4f}, "
    f"p = {spearman_p:.3e}"
)

Number of paired cells: 181124

Correlation between IDEAS ID-score and CCAT
---------------------------------------------
Pearson r = 0.2773, p = 0.000e+00
Spearman rho = 0.2818, p = 0.000e+00


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr

# =========================================================
# Complete dataset
# =========================================================

plot_df = adata.obs[
    ["IDEAS_score", "CCAT_score"]
].dropna()

# =========================================================
# Calculate correlation HERE
# =========================================================

pearson_r, pearson_p = pearsonr(
    plot_df["IDEAS_score"],
    plot_df["CCAT_score"]
)

spearman_r, spearman_p = spearmanr(
    plot_df["IDEAS_score"],
    plot_df["CCAT_score"]
)

print(f"Number of paired cells: {len(plot_df)}")

print("\nCorrelation between IDEAS ID-score and CCAT")
print("---------------------------------------------")
print(f"Pearson r  = {pearson_r:.4f}")
print(f"Pearson p  = {pearson_p:.3e}")
print(f"Spearman ρ = {spearman_r:.4f}")
print(f"Spearman p = {spearman_p:.3e}")


# =========================================================
# Plot
# =========================================================

plt.figure(figsize=(7, 6))

# Cells in background
sns.scatterplot(
    data=plot_df,
    x="IDEAS_score",
    y="CCAT_score",
    s=3,
    alpha=0.05,
    linewidth=0,
    zorder=1
)

# Regression line in front
sns.regplot(
    data=plot_df,
    x="IDEAS_score",
    y="CCAT_score",
    scatter=False,
    ci=95,
    line_kws={
        "linewidth": 3,
        "zorder": 10
    }
)

plt.xlabel("IDEAS ID-score")
plt.ylabel("CCAT score")

plt.title(
    f"IDEAS ID-score vs CCAT\n"
    f"Pearson r = {pearson_r:.3f}   "
    f"Spearman ρ = {spearman_r:.3f}"
)

plt.tight_layout()
plt.show()

Number of paired cells: 181124

Correlation between IDEAS ID-score and CCAT
---------------------------------------------
Pearson r  = 0.2773
Pearson p  = 0.000e+00
Spearman ρ = 0.2818
Spearman p = 0.000e+00


In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

hb = ax.hexbin(
    corr_df["IDEAS_ID_score"],
    corr_df["CCAT_score"],
    gridsize=70,
    mincnt=1,
    bins="log"
)

# Regression line
sns.regplot(
    data=corr_df,
    x="IDEAS_ID_score",
    y="CCAT_score",
    scatter=False,
    ci=95,
    line_kws={"linewidth": 2},
    ax=ax
)

ax.text(
    0.03,
    0.97,
    f"Pearson r = {pearson_r:.3f}\n"
    f"Spearman ρ = {spearman_rho:.3f}\n"
    f"n = {len(corr_df):,}",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=11,
    bbox=dict(
        facecolor="white",
        edgecolor="black",
        boxstyle="round,pad=0.4"
    )
)

ax.set_xlabel(
    "IDEAS ID-score",
    fontsize=13
)

ax.set_ylabel(
    "CCAT score",
    fontsize=13
)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# WOT trajectory definitions
# ============================================================

fates = {
    "iPSC": "w_iPSC",
    "Stromal": "w_Stromal",
    "Trophoblast": "w_Trophoblast",
    "Neural": "w_Neural"
}


# ============================================================
# Calculate WOT-weighted IDEAS trajectory
# ============================================================

results = []

for tp in sorted(adata.obs["timepoint"].unique()):

    if tp < 11:
        continue

    for fate, weight_col in fates.items():

        df = adata.obs[
            adata.obs["timepoint"] == tp
        ][
            ["IDEAS_score", weight_col]
        ].dropna()

        df = df[df[weight_col] > 0]

        if len(df) == 0:
            continue

        x = df["IDEAS_score"].to_numpy()
        w = df[weight_col].to_numpy()

        # WOT trajectory mean
        mean = np.average(
            x,
            weights=w
        )

        # Weighted SD
        sd = np.sqrt(
            np.average(
                (x - mean) ** 2,
                weights=w
            )
        )

        results.append({
            "timepoint": tp,
            "fate": fate,
            "IDEAS_mean": mean,
            "IDEAS_SD": sd
        })


ideas_wot = pd.DataFrame(results)


# ============================================================
# Publication-style WOT trend
# ============================================================

fig, ax = plt.subplots(
    figsize=(8, 5.5)
)


# Alternative trajectories
for fate in ["Stromal", "Trophoblast", "Neural"]:

    df = ideas_wot[
        ideas_wot["fate"] == fate
    ].sort_values("timepoint")

    x = df["timepoint"].to_numpy()
    y = df["IDEAS_mean"].to_numpy()
    sd = df["IDEAS_SD"].to_numpy()

    ax.plot(
        x,
        y,
        marker="o",
        markersize=3.5,
        linewidth=1.5,
        alpha=0.7,
        label=fate
    )

    # Extremely light SD
    ax.fill_between(
        x,
        y - sd,
        y + sd,
        alpha=0.025
    )


# iPSC trajectory — emphasized
df = ideas_wot[
    ideas_wot["fate"] == "iPSC"
].sort_values("timepoint")

x = df["timepoint"].to_numpy()
y = df["IDEAS_mean"].to_numpy()
sd = df["IDEAS_SD"].to_numpy()

ax.plot(
    x,
    y,
    marker="o",
    markersize=4.5,
    linewidth=2.8,
    label="iPSC"
)

ax.fill_between(
    x,
    y - sd,
    y + sd,
    alpha=0.04
)


# ============================================================
# Formatting
# ============================================================

ax.set_xlabel(
    "Reprogramming time (days)",
    fontsize=11
)

ax.set_ylabel(
    "WOT-weighted IDEAS score",
    fontsize=11
)

ax.set_title(
    "IDEAS trajectory during late reprogramming",
    fontsize=12
)

ax.legend(
    frameon=False,
    ncol=4,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.03)
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.tick_params(
    labelsize=9
)

plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import wilcoxon

# Pivot trajectory means
trajectory = ideas_wot.pivot(
    index="timepoint",
    columns="fate",
    values="IDEAS_mean"
).dropna()

# iPSC vs each alternative trajectory
comparisons = [
    "Stromal",
    "Trophoblast",
    "Neural"
]

for fate in comparisons:

    difference = (
        trajectory["iPSC"] -
        trajectory[fate]
    )

    stat, p = wilcoxon(
        difference,
        alternative="greater"
    )

    print(
        f"iPSC > {fate}: "
        f"W = {stat:.2f}, p = {p:.4g}"
    )

iPSC > Stromal: W = 120.00, p = 3.052e-05
iPSC > Trophoblast: W = 120.00, p = 3.052e-05
iPSC > Neural: W = 120.00, p = 3.052e-05


Spearman and pearson for all the cells after 11 days between pluripotency score and id score.

In [ ]:
from scipy.stats import pearsonr, spearmanr

# ---------------------------------------------------------
# Select cells from day 11 onward
# ---------------------------------------------------------

corr_df = adata.obs[
    adata.obs["timepoint"] >= 11
][
    ["IDEAS_score", "score_pluripotency"]
].dropna()

# ---------------------------------------------------------
# Pearson correlation
# ---------------------------------------------------------

pearson_r, pearson_p = pearsonr(
    corr_df["IDEAS_score"],
    corr_df["score_pluripotency"]
)

# ---------------------------------------------------------
# Spearman correlation
# ---------------------------------------------------------

spearman_r, spearman_p = spearmanr(
    corr_df["IDEAS_score"],
    corr_df["score_pluripotency"]
)

# ---------------------------------------------------------
# Results
# ---------------------------------------------------------

print(f"Number of paired cells: {len(corr_df)}")

print("\nCorrelation between IDEAS score and pluripotency score")
print("-------------------------------------------------------")

print(f"Pearson r  = {pearson_r:.4f}")
print(f"Pearson p  = {pearson_p:.3e}")

print(f"Spearman ρ = {spearman_r:.4f}")
print(f"Spearman p = {spearman_p:.3e}")

Number of paired cells: 58491

Correlation between IDEAS score and pluripotency score
-------------------------------------------------------
Pearson r  = 0.6088
Pearson p  = 0.000e+00
Spearman ρ = 0.4579
Spearman p = 0.000e+00


In [ ]:
print(adata)

AnnData object with n_obs × n_vars = 181124 × 27998
    obs: 'n_genes', 'n_counts', 'timepoint', 'condition', 'KS_score', 'KS_epi_detected', 'KS_mes_detected', 'GS76_score', 'score_hallmark_emt', 'score_emt_core', 'score_met_epithelial', 'score_pluripotency', 'score_fibroblast_id', 'score_bcell_id', 'score_epithelial_id', 'score_pluripotent_fate', 'score_stromal_fate', 'score_trophoblast_fate', 'score_neural_fate', 'score_ks_epithelial', 'score_ks_mesenchymal', 'orig_barcode', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'pct_counts_ribo', 'tp', 'IDEAS_score', 'w_iPSC', 'w_Stromal', 'w_Trophoblast', 'w_Neural', 'P_iPSC', 'P_Stromal', 'P_Trophoblast', 'P_Neural', 'P_Other', 'CCAT_score'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mt', 'ribo', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'
    uns: 'EMT_scoring', 'SCENT_CCAT', 'hvg', 'log1p', 'neighbors', 'pca', 'tp_colors', 'umap'


In [ ]:
from scipy.stats import pearsonr, spearmanr
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------------------------------------------------
# Extract the two KS scores
# ---------------------------------------------------------

corr_df = adata.obs[
    ["score_ks_epithelial", "score_ks_mesenchymal"]
].dropna()

# ---------------------------------------------------------
# Pearson correlation
# ---------------------------------------------------------

pearson_r, pearson_p = pearsonr(
    corr_df["score_ks_epithelial"],
    corr_df["score_ks_mesenchymal"]
)

# ---------------------------------------------------------
# Spearman correlation
# ---------------------------------------------------------

spearman_r, spearman_p = spearmanr(
    corr_df["score_ks_epithelial"],
    corr_df["score_ks_mesenchymal"]
)

# ---------------------------------------------------------
# Print results
# ---------------------------------------------------------

print(f"Number of paired cells: {len(corr_df)}")

print("\nCorrelation between KS epithelial and KS mesenchymal scores")
print("------------------------------------------------------------")

print(f"Pearson r  = {pearson_r:.4f}")
print(f"Pearson p  = {pearson_p:.3e}")

print(f"Spearman ρ = {spearman_r:.4f}")
print(f"Spearman p = {spearman_p:.3e}")

Number of paired cells: 181124

Correlation between KS epithelial and KS mesenchymal scores
------------------------------------------------------------
Pearson r  = -0.1225
Pearson p  = 0.000e+00
Spearman ρ = -0.1145
Spearman p = 0.000e+00


In [ ]:
plt.figure(figsize=(7, 6))

sns.scatterplot(
    data=corr_df,
    x="score_ks_epithelial",
    y="score_ks_mesenchymal",
    s=5,
    alpha=0.08,
    linewidth=0
)

plt.xlabel("KS epithelial score")
plt.ylabel("KS mesenchymal score")

plt.title(
    f"KS epithelial vs mesenchymal scores\n"
    f"Pearson r = {pearson_r:.3f}, "
    f"Spearman ρ = {spearman_r:.3f}"
)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr


# =========================================================
# Function to calculate correlations
# =========================================================

def calculate_ks_correlation(adata, mask, label):

    df = adata.obs.loc[
        mask,
        ["score_ks_epithelial", "score_ks_mesenchymal"]
    ].dropna()

    pearson_r, pearson_p = pearsonr(
        df["score_ks_epithelial"],
        df["score_ks_mesenchymal"]
    )

    spearman_r, spearman_p = spearmanr(
        df["score_ks_epithelial"],
        df["score_ks_mesenchymal"]
    )

    print("\n" + "=" * 65)
    print(label)
    print("=" * 65)

    print(f"Number of paired cells: {len(df)}")

    print(f"\nPearson:")
    print(f"r = {pearson_r:.4f}")
    print(f"p = {pearson_p:.3e}")

    print(f"\nSpearman:")
    print(f"rho = {spearman_r:.4f}")
    print(f"p = {spearman_p:.3e}")

    return df, pearson_r, pearson_p, spearman_r, spearman_p


# =========================================================
# EARLY REPROGRAMMING: before day 7.5
# =========================================================

early_df, early_pr, early_pp, early_sr, early_sp = \
    calculate_ks_correlation(
        adata,
        adata.obs["timepoint"] < 7.5,
        "EARLY REPROGRAMMING: < 7.5 days"
    )


# =========================================================
# LATE REPROGRAMMING: day 10 onward
# =========================================================

late_df, late_pr, late_pp, late_sr, late_sp = \
    calculate_ks_correlation(
        adata,
        adata.obs["timepoint"] >= 10,
        "LATE REPROGRAMMING: >= 10 days"
    )


EARLY REPROGRAMMING: < 7.5 days
Number of paired cells: 93136

Pearson:
r = -0.1331
p = 0.000e+00

Spearman:
rho = -0.0993
p = 8.576e-203

LATE REPROGRAMMING: >= 10 days
Number of paired cells: 62894

Pearson:
r = -0.2158
p = 0.000e+00

Spearman:
rho = -0.2218
p = 0.000e+00


In [ ]:
fig, axes = plt.subplots(
    1, 2,
    figsize=(13, 5.5)
)

# =========================================================
# EARLY
# =========================================================

axes[0].scatter(
    early_df["score_ks_epithelial"],
    early_df["score_ks_mesenchymal"],
    s=4,
    alpha=0.08
)

axes[0].set_xlabel("KS epithelial score")
axes[0].set_ylabel("KS mesenchymal score")

axes[0].set_title(
    f"Early reprogramming (<7.5 days)\n"
    f"Pearson r = {early_pr:.3f}   "
    f"Spearman ρ = {early_sr:.3f}"
)

# =========================================================
# LATE
# =========================================================

axes[1].scatter(
    late_df["score_ks_epithelial"],
    late_df["score_ks_mesenchymal"],
    s=4,
    alpha=0.08
)

axes[1].set_xlabel("KS epithelial score")
axes[1].set_ylabel("KS mesenchymal score")

axes[1].set_title(
    f"Late reprogramming (≥10 days)\n"
    f"Pearson r = {late_pr:.3f}   "
    f"Spearman ρ = {late_sr:.3f}"
)

# =========================================================
# Formatting
# =========================================================

for ax in axes:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# =========================================================
# Prepare data
# =========================================================

early_df = adata.obs[
    adata.obs["timepoint"] < 7.5
][
    ["score_ks_epithelial",
     "score_ks_mesenchymal",
     "timepoint"]
].dropna()


late_df = adata.obs[
    adata.obs["timepoint"] >= 10
][
    ["score_ks_epithelial",
     "score_ks_mesenchymal",
     "timepoint"]
].dropna()


# =========================================================
# Create shared color scale
# =========================================================

vmin = adata.obs["timepoint"].min()
vmax = adata.obs["timepoint"].max()

cmap = plt.cm.viridis


# =========================================================
# Plot
# =========================================================

fig, axes = plt.subplots(
    1, 2,
    figsize=(14, 5.5)
)


# =========================================================
# EARLY
# =========================================================

sc1 = axes[0].scatter(
    early_df["score_ks_epithelial"],
    early_df["score_ks_mesenchymal"],
    c=early_df["timepoint"],
    cmap=cmap,
    vmin=vmin,
    vmax=vmax,
    s=5,
    alpha=0.25,
    linewidth=0
)

axes[0].set_xlabel(
    "KS epithelial score"
)

axes[0].set_ylabel(
    "KS mesenchymal score"
)

axes[0].set_title(
    f"Early reprogramming (<7.5 days)\n"
    f"Pearson r = {early_pr:.3f}   "
    f"Spearman ρ = {early_sr:.3f}"
)


# =========================================================
# LATE
# =========================================================

axes[1].scatter(
    late_df["score_ks_epithelial"],
    late_df["score_ks_mesenchymal"],
    c=late_df["timepoint"],
    cmap=cmap,
    vmin=vmin,
    vmax=vmax,
    s=5,
    alpha=0.25,
    linewidth=0
)

axes[1].set_xlabel(
    "KS epithelial score"
)

axes[1].set_ylabel(
    "KS mesenchymal score"
)

axes[1].set_title(
    f"Late reprogramming (≥10 days)\n"
    f"Pearson r = {late_pr:.3f}   "
    f"Spearman ρ = {late_sr:.3f}"
)


# =========================================================
# Shared colorbar
# =========================================================

cbar = fig.colorbar(
    sc1,
    ax=axes,
    pad=0.02
)

cbar.set_label(
    "Reprogramming time (days)"
)


# =========================================================
# Clean publication-style formatting
# =========================================================

for ax in axes:

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.tick_params(
        labelsize=9
    )

plt.tight_layout()
plt.show()

/tmp/ipykernel_668838/518044623.py:137: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

# =========================================================
# 1. Prepare datasets
# =========================================================

cols = [
    "score_ks_epithelial",
    "score_ks_mesenchymal",
    "timepoint"
]

early_df = adata.obs[
    adata.obs["timepoint"] < 7.5
][cols].dropna()

late_df = adata.obs[
    adata.obs["timepoint"] >= 10
][cols].dropna()

all_df = adata.obs[
    cols
].dropna()


# =========================================================
# 2. Correlation function
# =========================================================

def get_correlations(df):

    pearson_r, pearson_p = pearsonr(
        df["score_ks_epithelial"],
        df["score_ks_mesenchymal"]
    )

    spearman_r, spearman_p = spearmanr(
        df["score_ks_epithelial"],
        df["score_ks_mesenchymal"]
    )

    return pearson_r, pearson_p, spearman_r, spearman_p


early_pr, early_pp, early_sr, early_sp = get_correlations(early_df)
late_pr, late_pp, late_sr, late_sp = get_correlations(late_df)
all_pr, all_pp, all_sr, all_sp = get_correlations(all_df)


# =========================================================
# 3. Print statistics
# =========================================================

print("=" * 70)
print("KS EPITHELIAL vs KS MESENCHYMAL")
print("=" * 70)

print("\nEARLY (<7.5 days)")
print(f"N = {len(early_df)}")
print(f"Pearson  r   = {early_pr:.4f}, p = {early_pp:.3e}")
print(f"Spearman rho = {early_sr:.4f}, p = {early_sp:.3e}")

print("\nLATE (≥10 days)")
print(f"N = {len(late_df)}")
print(f"Pearson  r   = {late_pr:.4f}, p = {late_pp:.3e}")
print(f"Spearman rho = {late_sr:.4f}, p = {late_sp:.3e}")

print("\nALL CELLS")
print(f"N = {len(all_df)}")
print(f"Pearson  r   = {all_pr:.4f}, p = {all_pp:.3e}")
print(f"Spearman rho = {all_sr:.4f}, p = {all_sp:.3e}")


# =========================================================
# 4. Create figure
# =========================================================

fig, axes = plt.subplots(
    1, 3,
    figsize=(18, 5.5)
)


# =========================================================
# PANEL 1 — EARLY
# =========================================================

sc1 = axes[0].scatter(
    early_df["score_ks_epithelial"],
    early_df["score_ks_mesenchymal"],
    c=early_df["timepoint"],
    cmap="viridis",
    vmin=early_df["timepoint"].min(),
    vmax=early_df["timepoint"].max(),
    s=5,
    alpha=0.18,
    linewidth=0
)

axes[0].set_title(
    "Early reprogramming (<7.5 days)\n"
    f"Pearson r = {early_pr:.3f}   "
    f"Spearman ρ = {early_sr:.3f}",
    fontsize=11
)

axes[0].set_xlabel(
    "KS epithelial score"
)

axes[0].set_ylabel(
    "KS mesenchymal score"
)

cbar1 = fig.colorbar(
    sc1,
    ax=axes[0],
    pad=0.02
)

cbar1.set_label(
    "Time (days)"
)


# =========================================================
# PANEL 2 — LATE
# =========================================================

sc2 = axes[1].scatter(
    late_df["score_ks_epithelial"],
    late_df["score_ks_mesenchymal"],
    c=late_df["timepoint"],
    cmap="viridis",
    vmin=late_df["timepoint"].min(),
    vmax=late_df["timepoint"].max(),
    s=5,
    alpha=0.18,
    linewidth=0
)

axes[1].set_title(
    "Late reprogramming (≥10 days)\n"
    f"Pearson r = {late_pr:.3f}   "
    f"Spearman ρ = {late_sr:.3f}",
    fontsize=11
)

axes[1].set_xlabel(
    "KS epithelial score"
)

axes[1].set_ylabel(
    "KS mesenchymal score"
)

cbar2 = fig.colorbar(
    sc2,
    ax=axes[1],
    pad=0.02
)

cbar2.set_label(
    "Time (days)"
)


# =========================================================
# PANEL 3 — ALL CELLS
# =========================================================

sc3 = axes[2].scatter(
    all_df["score_ks_epithelial"],
    all_df["score_ks_mesenchymal"],
    c=all_df["timepoint"],
    cmap="viridis",
    vmin=all_df["timepoint"].min(),
    vmax=all_df["timepoint"].max(),
    s=5,
    alpha=0.15,
    linewidth=0
)

axes[2].set_title(
    "All reprogramming timepoints\n"
    f"Pearson r = {all_pr:.3f}   "
    f"Spearman ρ = {all_sr:.3f}",
    fontsize=11
)

axes[2].set_xlabel(
    "KS epithelial score"
)

axes[2].set_ylabel(
    "KS mesenchymal score"
)

cbar3 = fig.colorbar(
    sc3,
    ax=axes[2],
    pad=0.02
)

cbar3.set_label(
    "Time (days)"
)


# =========================================================
# 5. Clean formatting
# =========================================================

for ax in axes:

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.tick_params(
        labelsize=9
    )

plt.tight_layout()
plt.show()

KS EPITHELIAL vs KS MESENCHYMAL

EARLY (<7.5 days)
N = 93136
Pearson  r   = -0.1331, p = 0.000e+00
Spearman rho = -0.0993, p = 8.576e-203

LATE (≥10 days)
N = 62894
Pearson  r   = -0.2158, p = 0.000e+00
Spearman rho = -0.2218, p = 0.000e+00

ALL CELLS
N = 181124
Pearson  r   = -0.1225, p = 0.000e+00
Spearman rho = -0.1145, p = 0.000e+00


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr

# ---------------------------------------------------------
# Cells after day 11
# ---------------------------------------------------------

plot_df = adata.obs[
    adata.obs["timepoint"] > 11
][
    ["IDEAS_score", "score_pluripotency"]
].dropna()

# ---------------------------------------------------------
# Correlations
# ---------------------------------------------------------

pearson_r, pearson_p = pearsonr(
    plot_df["IDEAS_score"],
    plot_df["score_pluripotency"]
)

spearman_r, spearman_p = spearmanr(
    plot_df["IDEAS_score"],
    plot_df["score_pluripotency"]
)

print(f"Number of paired cells: {len(plot_df)}")
print(f"Pearson r  = {pearson_r:.4f}, p = {pearson_p:.3e}")
print(f"Spearman ρ = {spearman_r:.4f}, p = {spearman_p:.3e}")


# ---------------------------------------------------------
# Plot
# ---------------------------------------------------------

plt.figure(figsize=(7, 6))

# Scatter
sns.scatterplot(
    data=plot_df,
    x="IDEAS_score",
    y="score_pluripotency",
    s=4,
    alpha=0.07,
    linewidth=0,
    zorder=1
)

# Regression line
sns.regplot(
    data=plot_df,
    x="IDEAS_score",
    y="score_pluripotency",
    scatter=False,
    ci=95,
    line_kws={
        "linewidth": 3,
        "zorder": 10
    }
)

plt.xlabel("IDEAS ID-score")
plt.ylabel("Pluripotency score")

plt.title(
    "IDEAS ID-score vs pluripotency score\n"
    "Cells after day 11\n"
    f"Pearson r = {pearson_r:.3f}   "
    f"Spearman ρ = {spearman_r:.3f}"
)

plt.tight_layout()
plt.show()

Number of paired cells: 55879
Pearson r  = 0.6135, p = 0.000e+00
Spearman ρ = 0.4712, p = 0.000e+00


In [ ]:
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# Keep day 11 onward
# ---------------------------------------------------------

plot_df = ideas_wot[
    ideas_wot["timepoint"] >= 11
].copy()


# ---------------------------------------------------------
# Plot
# ---------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(10, 6)
)

# Plot alternative trajectories first
for fate in ["Stromal", "Trophoblast", "Neural"]:

    df = plot_df[
        plot_df["fate"] == fate
    ].sort_values("timepoint")

    ax.plot(
        df["timepoint"],
        df["IDEAS_mean"],
        marker="o",
        markersize=4.5,
        linewidth=1.8,
        alpha=0.9,
        label=fate
    )


# ---------------------------------------------------------
# iPSC — emphasize
# ---------------------------------------------------------

df = plot_df[
    plot_df["fate"] == "iPSC"
].sort_values("timepoint")

ax.plot(
    df["timepoint"],
    df["IDEAS_mean"],
    marker="o",
    markersize=5,
    linewidth=2.8,
    alpha=1,
    label="iPSC"
)


# ---------------------------------------------------------
# Formatting
# ---------------------------------------------------------

ax.set_xlabel(
    "Reprogramming time (days)",
    fontsize=11
)

ax.set_ylabel(
    "WOT-weighted IDEAS score",
    fontsize=11
)

ax.set_title(
    "IDEAS score across Waddington-OT trajectories",
    fontsize=13
)

ax.legend(
    title="WOT fate",
    frameon=False
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.set_xlim(
    10.7,
    18.3
)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from scipy.stats import pearsonr

# Extract scores and retain early reprogramming cells only
df = adata.obs[
    ["timepoint", "score_ks_epithelial", "score_ks_mesenchymal"]
].copy()

df["day"] = pd.to_numeric(df["timepoint"], errors="coerce")

df = df[
    df["day"].between(0, 7.5)
].dropna(
    subset=[
        "day",
        "score_ks_epithelial",
        "score_ks_mesenchymal"
    ]
)

# Pearson correlation across all cells from days 0–7.5
pearson_r, pearson_p = pearsonr(
    df["score_ks_epithelial"],
    df["score_ks_mesenchymal"]
)

# Linear fit for the black trend line
slope, intercept = np.polyfit(
    df["score_ks_epithelial"],
    df["score_ks_mesenchymal"],
    1
)

x_line = np.linspace(
    df["score_ks_epithelial"].min(),
    df["score_ks_epithelial"].max(),
    200
)

y_line = slope * x_line + intercept

# Figure
fig, ax = plt.subplots(figsize=(9, 7))

# Each point is a cell; colour is its actual collection time
scatter = ax.scatter(
    df["score_ks_epithelial"],
    df["score_ks_mesenchymal"],
    c=df["day"],
    cmap="viridis",
    norm=Normalize(vmin=0, vmax=7.5),
    s=4,
    alpha=0.22,
    linewidths=0,
    rasterized=True
)

# Overall linear relationship
ax.plot(
    x_line,
    y_line,
    color="black",
    linewidth=2
)

# Continuous, linear time colour scale
cbar = fig.colorbar(scatter, ax=ax)
cbar.set_label("Reprogramming time (days)", fontsize=12)
cbar.set_ticks([0, 1, 2, 3, 4, 5, 6, 7, 7.5])

# Labels
ax.set_xlabel("KS epithelial score", fontsize=13)
ax.set_ylabel("KS mesenchymal score", fontsize=13)
ax.set_title("KS epithelial vs KS mesenchymal scores", fontsize=14)

# Pearson correlation and p-value
ax.text(
    0.03, 0.97,
    f"Days 0–7.5; n = {len(df):,}\n"
    f"Pearson r = {pearson_r:.3f}\n"
    f"p = {pearson_p:.2e}",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=10,
    bbox=dict(facecolor="white", edgecolor="none", alpha=0.9)
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from scipy.stats import pearsonr

# KS score and GS76 score from adata.obs
df = adata.obs[
    ["timepoint", "KS_score", "GS76_score"]
].copy()

df["day"] = pd.to_numeric(df["timepoint"], errors="coerce")

# Keep early reprogramming: days 0–7.5
df = df[
    df["day"].between(0, 7.5)
].dropna(
    subset=[
        "day",
        "KS_score",
        "GS76_score"
    ]
)

# Pearson correlation across all cells from days 0–7.5
pearson_r, pearson_p = pearsonr(
    df["KS_score"],
    df["GS76_score"]
)

# Linear-fit line
slope, intercept = np.polyfit(
    df["KS_score"],
    df["GS76_score"],
    1
)

x_line = np.linspace(
    df["KS_score"].min(),
    df["KS_score"].max(),
    200
)

y_line = slope * x_line + intercept

# Figure
fig, ax = plt.subplots(figsize=(9, 7))

# Each point is a cell; colour reflects actual collection time
scatter = ax.scatter(
    df["KS_score"],
    df["GS76_score"],
    c=df["day"],
    cmap="viridis",
    norm=Normalize(vmin=0, vmax=7.5),
    s=4,
    alpha=0.22,
    linewidths=0,
    rasterized=True
)

# Overall linear relationship
ax.plot(
    x_line,
    y_line,
    color="black",
    linewidth=2
)

# Continuous time colour bar
cbar = fig.colorbar(scatter, ax=ax)
cbar.set_label("Reprogramming time (days)", fontsize=12)
cbar.set_ticks([0, 1, 2, 3, 4, 5, 6, 7, 7.5])

# Labels
ax.set_xlabel("KS score", fontsize=13)
ax.set_ylabel("GS76 score", fontsize=13)
ax.set_title("KS score vs GS76 score", fontsize=14)

# Pearson correlation and p-value
ax.text(
    0.03, 0.97,
    f"Days 0–7.5; n = {len(df):,}\n"
    f"Pearson r = {pearson_r:.3f}\n"
    f"p = {pearson_p:.2e}",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=10,
    bbox=dict(facecolor="white", edgecolor="none", alpha=0.9)
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from scipy.stats import pearsonr

# All cells with both scores available
df = adata.obs[
    ["timepoint", "IDEAS_score", "CCAT_score"]
].copy()

df["day"] = pd.to_numeric(df["timepoint"], errors="coerce")

df = df.dropna(
    subset=["IDEAS_score", "CCAT_score"]
)

# Pearson correlation across the complete dataset
pearson_r, pearson_p = pearsonr(
    df["IDEAS_score"],
    df["CCAT_score"]
)

# Linear-fit line
slope, intercept = np.polyfit(
    df["IDEAS_score"],
    df["CCAT_score"],
    1
)

x_line = np.linspace(
    df["IDEAS_score"].min(),
    df["IDEAS_score"].max(),
    200
)

y_line = slope * x_line + intercept

fig, ax = plt.subplots(figsize=(9, 7))

# Cells are coloured by their collection time.
# Points without a numeric time point are shown in light grey.
has_day = df["day"].notna()

ax.scatter(
    df.loc[~has_day, "IDEAS_score"],
    df.loc[~has_day, "CCAT_score"],
    color="lightgrey",
    s=4,
    alpha=0.15,
    linewidths=0,
    rasterized=True
)

scatter = ax.scatter(
    df.loc[has_day, "IDEAS_score"],
    df.loc[has_day, "CCAT_score"],
    c=df.loc[has_day, "day"],
    cmap="viridis",
    norm=Normalize(
        vmin=df.loc[has_day, "day"].min(),
        vmax=df.loc[has_day, "day"].max()
    ),
    s=4,
    alpha=0.22,
    linewidths=0,
    rasterized=True
)

# Overall linear relationship
ax.plot(
    x_line,
    y_line,
    color="black",
    linewidth=2
)

# Time colour bar
cbar = fig.colorbar(scatter, ax=ax)
cbar.set_label("Reprogramming time (days)", fontsize=12)

# Relevant labels
ax.set_xlabel("IDEAS score (intrinsic dimension)", fontsize=13)
ax.set_ylabel("CCAT score (signaling entropy)", fontsize=13)
ax.set_title(
    "Signaling entropy versus intrinsic dimension: full dataset",
    fontsize=14
)

ax.text(
    0.03, 0.97,
    f"All cells; n = {len(df):,}\n"
    f"Pearson r = {pearson_r:.3f}\n"
    f"p = {pearson_p:.2e}",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=10,
    bbox=dict(facecolor="white", edgecolor="none", alpha=0.9)
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Same time-bin map as the violin plot
group_map = {
    0.0: "0–0.5", 0.5: "0–0.5",
    1.0: "1–1.5", 1.5: "1–1.5",
    2.0: "2–2.5", 2.5: "2–2.5",
    3.0: "3–3.5", 3.5: "3–3.5",
    4.0: "4–4.5", 4.5: "4–4.5",
    5.0: "5–5.5", 5.5: "5–5.5",
    6.0: "6–6.5", 6.5: "6–6.5",
    7.0: "7–7.5", 7.5: "7–7.5",
}

order = [
    "0–0.5", "1–1.5", "2–2.5", "3–3.5",
    "4–4.5", "5–5.5", "6–6.5", "7–7.5"
]

# Prepare IDEAS-score data
df = adata.obs[["timepoint", "IDEAS_score"]].copy()

df["day"] = pd.to_numeric(df["timepoint"], errors="coerce")

df = df[
    df["day"].between(0, 7.5)
].dropna(subset=["day", "IDEAS_score"])

df["time_group"] = df["day"].map(group_map)
df = df.dropna(subset=["time_group"])

# One-way ANOVA
groups = [
    df.loc[df["time_group"] == group, "IDEAS_score"]
    for group in order
]

F, p_anova = f_oneway(*groups)

print("One-way ANOVA: IDEAS score across time")
print(f"F = {F:.3f}")
print(f"p = {p_anova:.3e}")

# Tukey HSD pairwise comparisons
tukey = pairwise_tukeyhsd(
    endog=df["IDEAS_score"],
    groups=df["time_group"],
    alpha=0.05
)

# Turn Tukey output into a symmetric adjusted-p-value matrix
tukey_df = pd.DataFrame(
    tukey._results_table.data[1:],
    columns=tukey._results_table.data[0]
)

tukey_df["p-adj"] = tukey_df["p-adj"].astype(float)

p_matrix = pd.DataFrame(
    np.ones((len(order), len(order))),
    index=order,
    columns=order
)

for _, row in tukey_df.iterrows():
    group1 = row["group1"]
    group2 = row["group2"]
    p_value = row["p-adj"]

    p_matrix.loc[group1, group2] = p_value
    p_matrix.loc[group2, group1] = p_value

# Keep only lower triangle, as in the reference plot
mask = np.triu(
    np.ones_like(p_matrix, dtype=bool)
)

# Plot pairwise Tukey-comparison heatmap
fig, ax = plt.subplots(figsize=(9, 7))

sns.heatmap(
    p_matrix,
    mask=mask,
    cmap="viridis_r",
    vmin=0,
    vmax=0.05,
    annot=True,
    fmt=".2e",
    square=True,
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "Tukey-adjusted p-value"},
    ax=ax
)

ax.set_title(
    "IDEAS score: pairwise Tukey HSD comparisons",
    fontsize=13
)

ax.set_xlabel("Reprogramming time (days)")
ax.set_ylabel("Reprogramming time (days)")

plt.tight_layout()
plt.show()

One-way ANOVA: IDEAS score across time
F = 5114.062
p = 0.000e+00


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import f_oneway

# Same time bins
group_map = {
    0.0: "0–0.5", 0.5: "0–0.5",
    1.0: "1–1.5", 1.5: "1–1.5",
    2.0: "2–2.5", 2.5: "2–2.5",
    3.0: "3–3.5", 3.5: "3–3.5",
    4.0: "4–4.5", 4.5: "4–4.5",
    5.0: "5–5.5", 5.5: "5–5.5",
    6.0: "6–6.5", 6.5: "6–6.5",
    7.0: "7–7.5", 7.5: "7–7.5",
}

order = [
    "0–0.5", "1–1.5", "2–2.5", "3–3.5",
    "4–4.5", "5–5.5", "6–6.5", "7–7.5"
]

# Prepare data
df = adata.obs[["timepoint", "IDEAS_score"]].copy()

df["day"] = pd.to_numeric(df["timepoint"], errors="coerce")

df = df[
    df["day"].between(0, 7.5)
].dropna(subset=["day", "IDEAS_score"])

df["time_group"] = df["day"].map(group_map)
df = df.dropna(subset=["time_group"])

# One-way ANOVA across time bins
groups = [
    df.loc[df["time_group"] == group, "IDEAS_score"]
    for group in order
]

F, p_anova = f_oneway(*groups)

# Cell numbers for each time bin
cell_counts = (
    df.groupby("time_group", observed=True)
    .size()
    .reindex(order)
)

# Plot
fig, ax = plt.subplots(figsize=(12, 8))

# Violin distributions
sns.violinplot(
    data=df,
    x="time_group",
    y="IDEAS_score",
    order=order,
    inner=None,
    cut=0,
    linewidth=1.2,
    color="#3274A1",
    ax=ax
)

# Box plot: median and interquartile range
sns.boxplot(
    data=df,
    x="time_group",
    y="IDEAS_score",
    order=order,
    width=0.15,
    showfliers=False,
    boxprops=dict(
        facecolor="white",
        edgecolor="black",
        linewidth=1.1
    ),
    whiskerprops=dict(color="black", linewidth=1.1),
    capprops=dict(color="black", linewidth=1.1),
    medianprops=dict(color="black", linewidth=1.5),
    ax=ax
)

# Overall summary statistic
p_text = "p < 0.001" if p_anova < 0.001 else f"p = {p_anova:.3f}"

ax.text(
    0.02, 0.97,
    f"One-way ANOVA\nF = {F:.1f}, {p_text}",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=10,
    bbox=dict(
        facecolor="white",
        edgecolor="black",
        linewidth=0.8,
        boxstyle="round,pad=0.4"
    )
)

# Number of cells below each violin
for i, group in enumerate(order):
    ax.text(
        i,
        -0.075,
        f"n = {cell_counts.loc[group]:,}",
        transform=ax.get_xaxis_transform(),
        ha="center",
        va="top",
        fontsize=9
    )

# Labels and styling
ax.set_xlabel(
    "Reprogramming time (days)",
    fontsize=13,
    labelpad=38
)

ax.set_ylabel("IDEAS score", fontsize=13)
ax.set_title("IDEAS score across reprogramming time", fontsize=14)

ax.tick_params(axis="both", labelsize=11)

sns.despine(ax=ax)

# Extra space for the cell-number labels
plt.subplots_adjust(
    bottom=0.20,
    top=0.93,
    left=0.10,
    right=0.98
)

plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from scipy.stats import pearsonr

# Extract late-reprogramming cells: day 10.5 and later
df = adata.obs[
    ["timepoint", "IDEAS_score", "score_pluripotency"]
].copy()

df["day"] = pd.to_numeric(df["timepoint"], errors="coerce")

df = df[
    df["day"] >= 10.5
].dropna(
    subset=[
        "day",
        "IDEAS_score",
        "score_pluripotency"
    ]
)

# Pearson correlation
pearson_r, pearson_p = pearsonr(
    df["IDEAS_score"],
    df["score_pluripotency"]
)

# Linear-fit line
slope, intercept = np.polyfit(
    df["IDEAS_score"],
    df["score_pluripotency"],
    1
)

x_line = np.linspace(
    df["IDEAS_score"].min(),
    df["IDEAS_score"].max(),
    200
)

y_line = slope * x_line + intercept

# Figure
fig, ax = plt.subplots(figsize=(9, 7))

# Each point is a cell; colour represents collection time
scatter = ax.scatter(
    df["IDEAS_score"],
    df["score_pluripotency"],
    c=df["day"],
    cmap="viridis",
    norm=Normalize(
        vmin=df["day"].min(),
        vmax=df["day"].max()
    ),
    s=4,
    alpha=0.20,
    linewidths=0,
    rasterized=True
)

# Overall Pearson linear relationship
ax.plot(
    x_line,
    y_line,
    color="black",
    linewidth=2
)

# Continuous time colour scale
cbar = fig.colorbar(scatter, ax=ax)
cbar.set_label("Reprogramming time (days)", fontsize=12)

# Labels
ax.set_xlabel("IDEAS score (intrinsic dimension)", fontsize=13)
ax.set_ylabel("Pluripotency score", fontsize=13)
ax.set_title(
    "IDEAS score vs pluripotency score: late reprogramming",
    fontsize=14
)

# Pearson result
ax.text(
    0.03, 0.97,
    f"Days ≥10.5; n = {len(df):,}\n"
    f"Pearson r = {pearson_r:.3f}\n"
    f"p = {pearson_p:.2e}",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=10,
    bbox=dict(facecolor="white", edgecolor="none", alpha=0.9)
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from scipy.stats import pearsonr

df = adata.obs[
    ["timepoint", "score_ks_epithelial", "score_ks_mesenchymal"]
].copy()

df["day"] = pd.to_numeric(df["timepoint"], errors="coerce")

df = df[
    df["day"] >= 10.5
].dropna(
    subset=[
        "day",
        "score_ks_epithelial",
        "score_ks_mesenchymal"
    ]
)

pearson_r, pearson_p = pearsonr(
    df["score_ks_epithelial"],
    df["score_ks_mesenchymal"]
)

slope, intercept = np.polyfit(
    df["score_ks_epithelial"],
    df["score_ks_mesenchymal"],
    1
)

x_line = np.linspace(
    df["score_ks_epithelial"].min(),
    df["score_ks_epithelial"].max(),
    200
)

fig, ax = plt.subplots(figsize=(9, 7))

scatter = ax.scatter(
    df["score_ks_epithelial"],
    df["score_ks_mesenchymal"],
    c=df["day"],
    cmap="viridis",
    norm=Normalize(vmin=df["day"].min(), vmax=df["day"].max()),
    s=4,
    alpha=0.20,
    linewidths=0,
    rasterized=True
)

ax.plot(
    x_line,
    slope * x_line + intercept,
    color="black",
    linewidth=2
)

cbar = fig.colorbar(scatter, ax=ax)
cbar.set_label("Reprogramming time (days)", fontsize=12)

ax.set_xlabel("KS epithelial score", fontsize=13)
ax.set_ylabel("KS mesenchymal score", fontsize=13)
ax.set_title(
    "KS epithelial vs KS mesenchymal scores: late reprogramming",
    fontsize=14
)

ax.text(
    0.03, 0.97,
    f"Days ≥10.5; n = {len(df):,}\n"
    f"Pearson r = {pearson_r:.3f}\n"
    f"p = {pearson_p:.2e}",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=10,
    bbox=dict(facecolor="white", edgecolor="none", alpha=0.9)
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from scipy.stats import pearsonr

df = adata.obs[
    ["timepoint", "KS_score", "GS76_score"]
].copy()

df["day"] = pd.to_numeric(df["timepoint"], errors="coerce")

df = df[
    df["day"] >= 10.5
].dropna(
    subset=["day", "KS_score", "GS76_score"]
)

pearson_r, pearson_p = pearsonr(
    df["KS_score"],
    df["GS76_score"]
)

slope, intercept = np.polyfit(
    df["KS_score"],
    df["GS76_score"],
    1
)

x_line = np.linspace(
    df["KS_score"].min(),
    df["KS_score"].max(),
    200
)

fig, ax = plt.subplots(figsize=(9, 7))

scatter = ax.scatter(
    df["KS_score"],
    df["GS76_score"],
    c=df["day"],
    cmap="viridis",
    norm=Normalize(vmin=df["day"].min(), vmax=df["day"].max()),
    s=4,
    alpha=0.20,
    linewidths=0,
    rasterized=True
)

ax.plot(
    x_line,
    slope * x_line + intercept,
    color="black",
    linewidth=2
)

cbar = fig.colorbar(scatter, ax=ax)
cbar.set_label("Reprogramming time (days)", fontsize=12)

ax.set_xlabel("KS score", fontsize=13)
ax.set_ylabel("76GS score", fontsize=13)
ax.set_title(
    "KS score vs 76GS score: late reprogramming",
    fontsize=14
)

ax.text(
    0.03, 0.97,
    f"Days ≥10.5; n = {len(df):,}\n"
    f"Pearson r = {pearson_r:.3f}\n"
    f"p = {pearson_p:.2e}",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=10,
    bbox=dict(facecolor="white", edgecolor="none", alpha=0.9)
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import f_oneway

# Same time bins
group_map = {
    0.0: "0–0.5", 0.5: "0–0.5",
    1.0: "1–1.5", 1.5: "1–1.5",
    2.0: "2–2.5", 2.5: "2–2.5",
    3.0: "3–3.5", 3.5: "3–3.5",
    4.0: "4–4.5", 4.5: "4–4.5",
    5.0: "5–5.5", 5.5: "5–5.5",
    6.0: "6–6.5", 6.5: "6–6.5",
    7.0: "7–7.5", 7.5: "7–7.5",
}

order = [
    "0–0.5", "1–1.5", "2–2.5", "3–3.5",
    "4–4.5", "5–5.5", "6–6.5", "7–7.5"
]

# Data: IDEAS_score is the score column in your AnnData object
df = adata.obs[["timepoint", "IDEAS_score"]].copy()

df["day"] = pd.to_numeric(df["timepoint"], errors="coerce")

df = df[
    df["day"].between(0, 7.5)
].dropna(subset=["day", "IDEAS_score"])

df["time_group"] = df["day"].map(group_map)
df = df.dropna(subset=["time_group"])

# One-way ANOVA
groups = [
    df.loc[df["time_group"] == group, "IDEAS_score"]
    for group in order
]

F, p_anova = f_oneway(*groups)

# Cell counts and means
cell_counts = (
    df.groupby("time_group", observed=True)
    .size()
    .reindex(order)
)

group_means = (
    df.groupby("time_group", observed=True)["IDEAS_score"]
    .mean()
    .reindex(order)
)

# Plot
fig, ax = plt.subplots(figsize=(12, 8))

# Violins
sns.violinplot(
    data=df,
    x="time_group",
    y="IDEAS_score",
    order=order,
    inner=None,
    cut=0,
    linewidth=1.2,
    color="#3274A1",
    ax=ax
)

# White internal box plots
sns.boxplot(
    data=df,
    x="time_group",
    y="IDEAS_score",
    order=order,
    width=0.15,
    showfliers=False,
    boxprops=dict(
        facecolor="white",
        edgecolor="black",
        linewidth=1.1
    ),
    whiskerprops=dict(color="black", linewidth=1.1),
    capprops=dict(color="black", linewidth=1.1),
    medianprops=dict(color="black", linewidth=1.5),
    ax=ax
)

# Mean trajectory
ax.plot(
    np.arange(len(order)),
    group_means.values,
    color="#ff7f0e",
    linestyle="--",
    linewidth=1.8,
    marker="o",
    markersize=5,
    label="Mean trajectory",
    zorder=5
)

# Overall ANOVA result
p_text = "p < 0.001" if p_anova < 0.001 else f"p = {p_anova:.3f}"

ax.text(
    0.02, 0.97,
    f"One-way ANOVA\nF = {F:.1f}, {p_text}",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=10,
    bbox=dict(
        facecolor="white",
        edgecolor="black",
        linewidth=0.8,
        boxstyle="round,pad=0.4"
    )
)

# Cell counts below each time bin
for i, group in enumerate(order):
    ax.text(
        i,
        -0.075,
        f"n = {cell_counts.loc[group]:,}",
        transform=ax.get_xaxis_transform(),
        ha="center",
        va="top",
        fontsize=9
    )

# Labels: display ID rather than IDEAS
ax.set_xlabel(
    "Reprogramming time (days)",
    fontsize=13,
    labelpad=38
)

ax.set_ylabel("ID score", fontsize=13)
ax.set_title("ID score across reprogramming time", fontsize=14)

ax.legend(frameon=False, loc="upper right")

ax.tick_params(axis="both", labelsize=11)

sns.despine(ax=ax)

plt.subplots_adjust(
    bottom=0.20,
    top=0.93,
    left=0.10,
    right=0.98
)

plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import f_oneway

group_map = {
    0.0: "0–0.5", 0.5: "0–0.5",
    1.0: "1–1.5", 1.5: "1–1.5",
    2.0: "2–2.5", 2.5: "2–2.5",
    3.0: "3–3.5", 3.5: "3–3.5",
    4.0: "4–4.5", 4.5: "4–4.5",
    5.0: "5–5.5", 5.5: "5–5.5",
    6.0: "6–6.5", 6.5: "6–6.5",
    7.0: "7–7.5", 7.5: "7–7.5",
}

order = [
    "0–0.5", "1–1.5", "2–2.5", "3–3.5",
    "4–4.5", "5–5.5", "6–6.5", "7–7.5"
]

# Prepare ID-score data
df = adata.obs[["timepoint", "IDEAS_score"]].copy()

df["day"] = pd.to_numeric(df["timepoint"], errors="coerce")

df = df[
    df["day"].between(0, 7.5)
].dropna(subset=["day", "IDEAS_score"])

df["time_group"] = df["day"].map(group_map)
df = df.dropna(subset=["time_group"])

# Min–max normalization: ID score becomes 0–1
id_min = df["IDEAS_score"].min()
id_max = df["IDEAS_score"].max()

df["ID_score_normalized"] = (
    (df["IDEAS_score"] - id_min) /
    (id_max - id_min)
)

# One-way ANOVA
groups = [
    df.loc[
        df["time_group"] == group,
        "ID_score_normalized"
    ]
    for group in order
]

F, p_anova = f_oneway(*groups)

# Counts and means
cell_counts = (
    df.groupby("time_group", observed=True)
    .size()
    .reindex(order)
)

group_means = (
    df.groupby("time_group", observed=True)["ID_score_normalized"]
    .mean()
    .reindex(order)
)

# Plot
fig, ax = plt.subplots(figsize=(12, 8))

sns.violinplot(
    data=df,
    x="time_group",
    y="ID_score_normalized",
    order=order,
    inner=None,
    cut=0,
    linewidth=1.2,
    color="#3274A1",
    ax=ax
)

sns.boxplot(
    data=df,
    x="time_group",
    y="ID_score_normalized",
    order=order,
    width=0.15,
    showfliers=False,
    boxprops=dict(
        facecolor="white",
        edgecolor="black",
        linewidth=1.1
    ),
    whiskerprops=dict(color="black", linewidth=1.1),
    capprops=dict(color="black", linewidth=1.1),
    medianprops=dict(color="black", linewidth=1.5),
    ax=ax
)

# Mean trajectory
ax.plot(
    np.arange(len(order)),
    group_means.values,
    color="#ff7f0e",
    linestyle="--",
    linewidth=1.8,
    marker="o",
    markersize=5,
    label="Mean trajectory",
    zorder=5
)

# ANOVA summary
p_text = "p < 0.001" if p_anova < 0.001 else f"p = {p_anova:.3f}"

ax.text(
    0.02, 0.97,
    f"One-way ANOVA\nF = {F:.1f}, {p_text}",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=10,
    bbox=dict(
        facecolor="white",
        edgecolor="black",
        linewidth=0.8,
        boxstyle="round,pad=0.4"
    )
)

# Cell counts
for i, group in enumerate(order):
    ax.text(
        i,
        -0.075,
        f"n = {cell_counts.loc[group]:,}",
        transform=ax.get_xaxis_transform(),
        ha="center",
        va="top",
        fontsize=9
    )

ax.set_xlabel(
    "Reprogramming time (days)",
    fontsize=13,
    labelpad=38
)

ax.set_ylabel("ID score", fontsize=13)
ax.set_title("ID score across reprogramming time", fontsize=14)

ax.set_ylim(0, 1)
ax.set_yticks(np.arange(0, 1.01, 0.2))

ax.legend(frameon=False, loc="upper right")

sns.despine(ax=ax)

plt.subplots_adjust(
    bottom=0.20,
    top=0.93,
    left=0.10,
    right=0.98
)

plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import f_oneway

# Same time bins
group_map = {
    0.0: "0–0.5", 0.5: "0–0.5",
    1.0: "1–1.5", 1.5: "1–1.5",
    2.0: "2–2.5", 2.5: "2–2.5",
    3.0: "3–3.5", 3.5: "3–3.5",
    4.0: "4–4.5", 4.5: "4–4.5",
    5.0: "5–5.5", 5.5: "5–5.5",
    6.0: "6–6.5", 6.5: "6–6.5",
    7.0: "7–7.5", 7.5: "7–7.5",
}

order = [
    "0–0.5", "1–1.5", "2–2.5", "3–3.5",
    "4–4.5", "5–5.5", "6–6.5", "7–7.5"
]

# Data
df = adata.obs[["timepoint", "CCAT_score"]].copy()

df["day"] = pd.to_numeric(df["timepoint"], errors="coerce")

df = df[
    df["day"].between(0, 7.5)
].dropna(subset=["day", "CCAT_score"])

df["time_group"] = df["day"].map(group_map)
df = df.dropna(subset=["time_group"])

# One-way ANOVA
groups = [
    df.loc[df["time_group"] == group, "CCAT_score"]
    for group in order
]

F, p_anova = f_oneway(*groups)

# Cell counts and means
cell_counts = (
    df.groupby("time_group", observed=True)
    .size()
    .reindex(order)
)

group_means = (
    df.groupby("time_group", observed=True)["CCAT_score"]
    .mean()
    .reindex(order)
)

# Plot
fig, ax = plt.subplots(figsize=(12, 8))

# Violin distributions
sns.violinplot(
    data=df,
    x="time_group",
    y="CCAT_score",
    order=order,
    inner=None,
    cut=0,
    linewidth=1.2,
    color="#3274A1",
    ax=ax
)

# White internal box plots
sns.boxplot(
    data=df,
    x="time_group",
    y="CCAT_score",
    order=order,
    width=0.15,
    showfliers=False,
    boxprops=dict(
        facecolor="white",
        edgecolor="black",
        linewidth=1.1
    ),
    whiskerprops=dict(color="black", linewidth=1.1),
    capprops=dict(color="black", linewidth=1.1),
    medianprops=dict(color="black", linewidth=1.5),
    ax=ax
)

# Mean trajectory
ax.plot(
    np.arange(len(order)),
    group_means.values,
    color="#ff7f0e",
    linestyle="--",
    linewidth=1.8,
    marker="o",
    markersize=5,
    label="Mean trajectory",
    zorder=5
)

# Overall ANOVA result
p_text = "p < 0.001" if p_anova < 0.001 else f"p = {p_anova:.3f}"

ax.text(
    0.02, 0.97,
    f"One-way ANOVA\nF = {F:.1f}, {p_text}",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=10,
    bbox=dict(
        facecolor="white",
        edgecolor="black",
        linewidth=0.8,
        boxstyle="round,pad=0.4"
    )
)

# Cell numbers
for i, group in enumerate(order):
    ax.text(
        i,
        -0.075,
        f"n = {cell_counts.loc[group]:,}",
        transform=ax.get_xaxis_transform(),
        ha="center",
        va="top",
        fontsize=9
    )

# Labels
ax.set_xlabel(
    "Reprogramming time (days)",
    fontsize=13,
    labelpad=38
)

ax.set_ylabel("Signalling Entropy score", fontsize=13)
ax.set_title(
    "Signalling Entropy score across reprogramming time",
    fontsize=14
)

ax.legend(frameon=False, loc="upper right")

ax.tick_params(axis="both", labelsize=11)

sns.despine(ax=ax)

plt.subplots_adjust(
    bottom=0.20,
    top=0.93,
    left=0.10,
    right=0.98
)

plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd

group_map = {
    0.0: "0–0.5", 0.5: "0–0.5",
    1.0: "1–1.5", 1.5: "1–1.5",
    2.0: "2–2.5", 2.5: "2–2.5",
    3.0: "3–3.5", 3.5: "3–3.5",
    4.0: "4–4.5", 4.5: "4–4.5",
    5.0: "5–5.5", 5.5: "5–5.5",
    6.0: "6–6.5", 6.5: "6–6.5",
    7.0: "7–7.5", 7.5: "7–7.5",
}

order = [
    "0–0.5", "1–1.5", "2–2.5", "3–3.5",
    "4–4.5", "5–5.5", "6–6.5", "7–7.5"
]

# Signaling-entropy data
df = adata.obs[["timepoint", "CCAT_score"]].copy()

df["day"] = pd.to_numeric(df["timepoint"], errors="coerce")

df = df[
    df["day"].between(0, 7.5)
].dropna(subset=["day", "CCAT_score"])

df["time_group"] = df["day"].map(group_map)
df = df.dropna(subset=["time_group"])

# One-way ANOVA
groups = [
    df.loc[df["time_group"] == group, "CCAT_score"]
    for group in order
]

F, p_anova = f_oneway(*groups)

print("One-way ANOVA: signaling entropy across time")
print(f"F = {F:.3f}")
print(f"p = {p_anova:.3e}")

# Tukey HSD pairwise comparisons
tukey = pairwise_tukeyhsd(
    endog=df["CCAT_score"],
    groups=df["time_group"],
    alpha=0.05
)

tukey_df = pd.DataFrame(
    tukey._results_table.data[1:],
    columns=tukey._results_table.data[0]
)

tukey_df["p-adj"] = tukey_df["p-adj"].astype(float)

p_matrix = pd.DataFrame(
    np.ones((len(order), len(order))),
    index=order,
    columns=order
)

for _, row in tukey_df.iterrows():
    p_matrix.loc[row["group1"], row["group2"]] = row["p-adj"]
    p_matrix.loc[row["group2"], row["group1"]] = row["p-adj"]

# Plot only the lower triangle
mask = np.triu(np.ones_like(p_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(9, 7))

sns.heatmap(
    p_matrix,
    mask=mask,
    cmap="viridis_r",
    vmin=0,
    vmax=0.05,
    annot=True,
    fmt=".2e",
    square=True,
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "Tukey-adjusted p-value"},
    ax=ax
)

ax.set_title(
    "Signaling entropy: pairwise Tukey HSD comparisons",
    fontsize=13
)

ax.set_xlabel("Reprogramming time (days)")
ax.set_ylabel("Reprogramming time (days)")

plt.tight_layout()
plt.show()

One-way ANOVA: signaling entropy across time
F = 2520.955
p = 0.000e+00


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Waddington-OT fate-weight columns
fates = {
    "iPSC": "w_iPSC",
    "Stromal": "w_Stromal",
    "Trophoblast": "w_Trophoblast",
    "Neural": "w_Neural"
}

# Use late reprogramming time points: day 10.5 and later
obs = adata.obs.copy()

obs["day"] = pd.to_numeric(obs["timepoint"], errors="coerce")
obs = obs[obs["day"] >= 10.5].copy()

# Function to calculate WOT-weighted mean score for each fate and day
def calculate_wot_trajectory(data, score_column, fate_dict):

    results = []

    for day in sorted(data["day"].dropna().unique()):

        day_data = data[data["day"] == day]

        for fate, weight_column in fate_dict.items():

            tmp = day_data[
                [score_column, weight_column]
            ].dropna()

            # Keep cells with non-zero WOT weight for this fate
            tmp = tmp[tmp[weight_column] > 0]

            if len(tmp) == 0:
                continue

            score_values = tmp[score_column].to_numpy()
            weights = tmp[weight_column].to_numpy()

            weighted_mean = np.average(
                score_values,
                weights=weights
            )

            results.append({
                "day": day,
                "fate": fate,
                "weighted_mean": weighted_mean
            })

    return pd.DataFrame(results)


# Calculate trajectories
id_wot = calculate_wot_trajectory(
    obs,
    score_column="IDEAS_score",
    fate_dict=fates
)

signaling_entropy_wot = calculate_wot_trajectory(
    obs,
    score_column="CCAT_score",
    fate_dict=fates
)


# Plotting function
def plot_wot_trajectories(
    trajectory_df,
    title,
    y_label
):

    fig, ax = plt.subplots(figsize=(10, 5))

    for fate in ["iPSC", "Stromal", "Trophoblast", "Neural"]:

        tmp = trajectory_df[
            trajectory_df["fate"] == fate
        ].sort_values("day")

        ax.plot(
            tmp["day"],
            tmp["weighted_mean"],
            marker="o",
            markersize=4,
            linewidth=1.8,
            label=fate
        )

    ax.set_title(title, fontsize=13)
    ax.set_xlabel("Reprogramming time (days)", fontsize=11)
    ax.set_ylabel(y_label, fontsize=11)

    ax.legend(
        title="WOT trajectory",
        frameon=False,
        loc="upper left"
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.tick_params(axis="x", rotation=45, labelsize=9)

    plt.tight_layout()
    plt.show()


# ID-score trajectory plot
plot_wot_trajectories(
    id_wot,
    title="ID score along Waddington-OT trajectories",
    y_label="WOT-weighted ID score"
)

# Signaling-entropy trajectory plot
plot_wot_trajectories(
    signaling_entropy_wot,
    title="Signaling entropy score along Waddington-OT trajectories",
    y_label="WOT-weighted signaling entropy score"
)

In [ ]:
import matplotlib.pyplot as plt

# Combine the two WOT-weighted score tables by fate and time
ks_wot = ks_epithelial_wot.merge(
    ks_mesenchymal_wot,
    on=["day", "fate"],
    suffixes=("_epithelial", "_mesenchymal")
)

fig, ax = plt.subplots(figsize=(8, 7))

for fate in ["iPSC", "Stromal", "Trophoblast", "Neural"]:

    tmp = ks_wot[
        ks_wot["fate"] == fate
    ].sort_values("day")

    # Line connects time points in chronological order
    ax.plot(
        tmp["weighted_mean_epithelial"],
        tmp["weighted_mean_mesenchymal"],
        marker="o",
        markersize=4,
        linewidth=2,
        label=fate
    )

    # Arrow indicates the direction of time
    ax.annotate(
        "",
        xy=(
            tmp["weighted_mean_epithelial"].iloc[-1],
            tmp["weighted_mean_mesenchymal"].iloc[-1]
        ),
        xytext=(
            tmp["weighted_mean_epithelial"].iloc[-2],
            tmp["weighted_mean_mesenchymal"].iloc[-2]
        ),
        arrowprops=dict(arrowstyle="->", lw=1.5)
    )

ax.set_xlabel("WOT-weighted KS epithelial score", fontsize=12)
ax.set_ylabel("WOT-weighted KS mesenchymal score", fontsize=12)

ax.set_title(
    "KS epithelial vs KS mesenchymal along Waddington-OT trajectories",
    fontsize=13
)

ax.legend(
    title="WOT trajectory",
    frameon=False
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()